# Machine Learning for Heart Disease Classification in a Northern Bangladesh Hospital Cohort

**Parent project:** AI-Driven Web-Based Heart Disease Prediction System Using Machine Learning

---

## 1. Research Context

Cardiovascular disease is the leading cause of mortality in Bangladesh, and diagnostic capacity is
unevenly distributed between urban tertiary centres and district-level facilities. Machine learning
applied to routinely collected admission data — demographics, vital signs and standard laboratory
panels — has been proposed as a means of supporting triage in resource-constrained settings, since
these variables are already recorded during normal clinical workflow and require no additional
investigation.

Most published models in this area are developed on long-standing public datasets, principally the
UCI Cleveland cohort, which reflect the case mix, measurement practices and population
characteristics of high-income settings recorded several decades ago. Whether comparable
performance is attainable on contemporary South Asian hospital records is not established.

This notebook develops and internally validates a supervised classification model for heart disease
status using a hospital-sourced dataset collected in northern Bangladesh. It documents the complete
analytical pipeline: data quality assessment, harmonisation of censored laboratory measurements,
leakage-controlled preprocessing, nested cross-validation, statistical comparison of candidate
classifiers, calibration assessment, and model explanation.

> **Scope statement.** The model developed here is a research prototype intended to support
> methodological investigation. It has not undergone external validation, prospective evaluation or
> regulatory assessment, and it is not a clinical diagnostic tool. Section 30 sets out the
> limitations in full.

## 2. Research Objective and Analysis Plan

### 2.1 Objective

To develop and internally validate a supervised classifier that distinguishes patients with and
without heart disease using demographic, anthropometric, vital-sign and routine laboratory variables
recorded at hospital admission, and to identify which of those variables carry the greatest
discriminative weight.

### 2.2 Research questions

1. What discriminative performance is attainable on this cohort using routinely collected admission
   variables, estimated without optimistic bias?
2. Do the candidate classifiers differ from one another to a statistically detectable degree, or are
   observed differences attributable to sampling variation?
3. Are the model's predicted probabilities calibrated well enough to be reported to a user, as
   distinct from merely being correctly ranked?
4. Which variables drive the predictions, and are their contributions consistent with established
   cardiovascular pathophysiology?

### 2.3 Pre-specified analysis plan

The following decisions are fixed before results are examined, so that model selection cannot be
influenced by held-out performance.

| Element | Specification |
|---|---|
| Outcome | `Heart_Disease`, binary (1 = heart disease present) |
| Population | Adult records (age ≥ 18 years) |
| Candidate models | Logistic regression, decision tree, support vector machine, random forest |
| Primary metric | ROC-AUC |
| Performance estimation | Nested cross-validation (5 outer folds × 5 inner folds), stratified |
| Model comparison | DeLong test on pooled out-of-fold predictions from the outer loop |
| Selection rule | Highest nested-CV AUC; where models are statistically indistinguishable from the leader, the most parsimonious member of that set is selected |
| Parsimony ordering | Logistic regression < decision tree < support vector machine < random forest |
| Held-out test set | 20%, stratified, used exclusively for final reporting — never for selection |
| Operating threshold | Reported at 0.50 and at a sensitivity-oriented threshold determined on training data only |

The parsimony rule reflects the deployment context. The parent project targets a web-based
application, where a smaller model with directly reportable coefficients is preferable to a larger
ensemble if discriminative performance is equivalent.

## 3. Dataset Provenance, Ethics and Data Dictionary

> ### ⚠️ Section requiring completion before submission
>
> The fields marked `[TO BE COMPLETED]` cannot be derived from the data file and must be supplied
> from the data-sharing agreement and ethics documentation. They are mandatory for supervisor
> submission and for any journal submission. **Nothing in this section has been inferred or
> assumed.**

### 3.1 Provenance

| Item | Value |
|---|---|
| Source institution(s) | `[TO BE COMPLETED]` |
| Geographic region | Northern Bangladesh |
| Collection period | `[TO BE COMPLETED]` |
| Sampling frame | `[TO BE COMPLETED — consecutive admissions, retrospective chart review, or other]` |
| Records supplied | 1,048 |
| Unit of observation | `[TO BE COMPLETED — confirm one record per unique patient]` |
| Data custodian | `[TO BE COMPLETED]` |

### 3.2 Ethics and governance

| Item | Value |
|---|---|
| Ethics/IRB approval body | `[TO BE COMPLETED]` |
| Approval reference number | `[TO BE COMPLETED]` |
| Consent procedure | `[TO BE COMPLETED]` |
| De-identification method | `[TO BE COMPLETED]` |
| Data-sharing agreement | `[TO BE COMPLETED]` |

### 3.3 Outcome variable definition

> **This is the single most important outstanding item.** The interpretation of every result in this
> notebook depends on it, and it must be obtained in writing from the data custodian.

| Item | Value |
|---|---|
| How was `Heart Disease` determined? | `[TO BE COMPLETED — e.g. angiographic stenosis threshold, discharge ICD code, clinician adjudication]` |
| Who assigned the label? | `[TO BE COMPLETED]` |
| When was it assigned relative to the laboratory tests? | `[TO BE COMPLETED]` |
| Were any of the predictor variables used to assign the label? | `[TO BE COMPLETED]` |

The final question is decisive. If lipid, glucose or troponin values contributed to the diagnostic
decision, then those variables are partially constitutive of the outcome rather than independent
predictors of it, and the model is recovering a labelling rule rather than predicting disease.
Section 12.3 reports diagnostic evidence bearing on this question.

**Project-specific note.** For this project the data custodian is not accessible: supervisory
involvement is limited to reviewing the completed work, not to supplying provenance information that
was never recorded alongside the dataset. This table therefore cannot be completed as originally
specified, and that fact is itself reported as a limitation (Section 30.1) rather than resolved by
assumption. Section 26.2 substitutes an empirical sensitivity analysis — refitting the selected model
without the variables implicated by Section 12.3 — as the closest available substitute for a
custodian confirmation, and its result should be read as bounding this risk, not eliminating it.

### 3.4 Data dictionary

Units marked † are inferred from the value ranges and conventional clinical reporting practice, and
require confirmation from the data custodian. They are not documented in the source file.

| Variable | Description | Type | Unit |
|---|---|---|---|
| `SL` | Record serial number | Identifier | — |
| `Age` | Age at admission | Continuous | years |
| `Sex` | Recorded sex | Categorical | M / F |
| `Height (cm)` | Height | Continuous | cm |
| `Weight (kg)` | Weight | Continuous | kg |
| `BMI` | Body mass index | Continuous | kg/m² |
| `Family H/O` | Family history of cardiovascular disease | Binary | 0 / 1 |
| `Hypertension` | Diagnosed hypertension | Binary | 0 / 1 |
| `Diabetes` | Diagnosed diabetes mellitus | Binary | 0 / 1 |
| `Total_Cholesterol(mg/dL)` | Total cholesterol | Continuous | mg/dL |
| `BP(mmHg)` | Blood pressure — **component unspecified** `[TO BE COMPLETED]` | Continuous | mmHg |
| `H/O ChestPain` | History of chest pain | Binary | 0 / 1 |
| `RBS(mmol/L)` | Random blood sugar | Continuous | mmol/L |
| `HDL(mg/dL)` | HDL cholesterol | Continuous | mg/dL |
| `LDL(mg/dL)` | LDL cholesterol | Continuous | mg/dL |
| `Triglycerides(mg/dL)` | Triglycerides | Continuous | mg/dL |
| `MaxHR` | Maximum heart rate — **measured or age-derived?** `[TO BE COMPLETED]` | Continuous | bpm † |
| `Himoglobin` | Haemoglobin (spelling as supplied) | Continuous | g/dL † |
| `Creatinine(mg/dL)` | Serum creatinine | Continuous | mg/dL |
| `Platelets` | Platelet count | Continuous | /µL † |
| `Sodium(mmol/L)` | Serum sodium | Continuous | mmol/L |
| `Potassium` | Serum potassium | Continuous | mmol/L † |
| `Chloride` | Serum chloride | Continuous | mmol/L † |
| `Troponin-I` | Cardiac troponin I | Continuous | assay-dependent |
| `Troponin- I assay type` | Assay platform | Categorical | conventional / high-sensitivity |
| `Heart Disease` | **Outcome** | Binary | 0 / 1 |
| `UNIT` | Admitting ward | Categorical | CCU / General |

## 4. Computational Environment and Reproducibility

All stochastic components — the train/test partition, cross-validation fold assignment, estimator
initialisation, bootstrap resampling and SHAP background sampling — are seeded from a single
constant. Library versions resolved at run time are captured in Section 28 and written to the
metadata record, so that a future re-run can pin the exact stack that produced the reported results.

Warning suppression is applied narrowly. Convergence and numerical warnings are left visible,
because a silent `ConvergenceWarning` during hyperparameter search would otherwise go undetected.

In [ ]:
import sys, os, json, hashlib, platform, warnings, subprocess
from datetime import datetime, timezone

# Deprecation noise is suppressed; convergence and numerical warnings remain visible.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "openpyxl", "shap", "joblib"],
    check=False
)

import numpy as np
import pandas as pd
import scipy
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, brier_score_loss, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, classification_report
)

RANDOM_STATE = 42
N_BOOTSTRAP = 2000          # bootstrap replicates for confidence intervals
TEST_SIZE = 0.20
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 5
ALPHA = 0.05                # significance level for model comparison

np.random.seed(RANDOM_STATE)

# ---------------------------------------------------------------------------
# Configuration flags for changes that are pending supervisor approval.
# Each is accompanied by a diagnostic in the section referenced below.
# Defaults preserve the original methodology; flip only after reviewing evidence.
# ---------------------------------------------------------------------------
DROP_MAXHR = False               # See Section 10 diagnostic before changing.
USE_MISSINGNESS_INDICATORS = True  # See Section 15 diagnostic before changing.

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "font.size": 10,
})

OUTPUT_DIR = "northern_bangladesh_heart_disease_artifacts"
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
TAB_DIR = os.path.join(OUTPUT_DIR, "tables")
for d in (OUTPUT_DIR, FIG_DIR, TAB_DIR):
    os.makedirs(d, exist_ok=True)


def save_figure(name):
    '''Persist the current figure at publication resolution.'''
    plt.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=300)
    plt.savefig(os.path.join(FIG_DIR, f"{name}.pdf"))


def save_table(frame, name):
    '''Persist a results table and return it unchanged for display.'''
    frame.to_csv(os.path.join(TAB_DIR, f"{name}.csv"), index=True)
    return frame


ENVIRONMENT = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": __import__("matplotlib").__version__,
    "seaborn": sns.__version__,
    "joblib": joblib.__version__,
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "random_state": RANDOM_STATE,
}

print("Computational environment")
print("-" * 60)
for k, v in ENVIRONMENT.items():
    print(f"{k:<22} {v}")
print("-" * 60)
print("Pin these versions when re-running for archival reproducibility.")

## 5. Data Loading and File Integrity

The source file is hashed on load. Recording the SHA-256 digest alongside the results establishes
which exact version of the dataset produced them, which matters because clinical datasets are
frequently revised after initial release.

In [ ]:
EXPECTED = "Heart_diasease_dataset_from_Northern_Bangladesh.xlsx"

if os.path.exists(EXPECTED):
    DATA_PATH = EXPECTED
else:
    try:
        from google.colab import files  # Present only in Google Colab.
    except ImportError as exc:
        raise FileNotFoundError(
            f"'{EXPECTED}' was not found in the working directory. "
            "Place the file alongside this notebook, or run the notebook in Google Colab "
            "where it can be uploaded interactively."
        ) from exc

    print(f"Upload the dataset file: {EXPECTED}")
    uploaded = files.upload()
    excel_files = [n for n in uploaded if n.lower().endswith((".xlsx", ".xls"))]
    if not excel_files:
        raise FileNotFoundError("No Excel file was uploaded.")
    DATA_PATH = excel_files[0]

with open(DATA_PATH, "rb") as handle:
    DATA_SHA256 = hashlib.sha256(handle.read()).hexdigest()

workbook = pd.ExcelFile(DATA_PATH)
SHEET = "Our Dataset" if "Our Dataset" in workbook.sheet_names else workbook.sheet_names[0]
df_raw = pd.read_excel(DATA_PATH, sheet_name=SHEET)

print(f"File            : {DATA_PATH}")
print(f"SHA-256         : {DATA_SHA256}")
print(f"Sheets available: {workbook.sheet_names}")
print(f"Sheet loaded    : {SHEET}")
print(f"Dimensions      : {df_raw.shape[0]} records x {df_raw.shape[1]} columns")

display(df_raw.head())

## 6. Data Quality Assessment

Three properties are assessed before any transformation: record uniqueness, completeness, and the
distinction between variables stored with an appropriate data type and those stored as text.

### 6.1 Record uniqueness

Duplicate detection must exclude the serial number `SL`, which is unique by construction and would
otherwise guarantee that no duplicates are ever found. Two checks are reported: exact duplication
across all substantive columns, and duplication across clinical variables only. The latter is the
more informative of the two, because a patient recorded twice under different serial numbers would
produce identical clinical values.

This matters beyond data hygiene. If the same patient contributes multiple records, random
partitioning places correlated observations on both sides of the train/test boundary and inflates
every performance estimate reported downstream.

In [ ]:
n_records = len(df_raw)
substantive = [c for c in df_raw.columns if c != "SL"]
clinical = [c for c in substantive if c not in ("Heart Disease", "UNIT")]

dup_substantive = int(df_raw[substantive].duplicated().sum())
dup_clinical = int(df_raw[clinical].duplicated().sum())

print("Record uniqueness")
print("-" * 60)
print(f"Total records                                : {n_records}")
print(f"Unique values of SL                          : {df_raw['SL'].nunique()}")
print(f"Duplicates across all non-identifier columns  : {dup_substantive}")
print(f"Duplicates across clinical columns only       : {dup_clinical}")
print("-" * 60)

if dup_clinical > 0:
    print(
        f"WARNING: {dup_clinical} record(s) share identical clinical values with another record.\n"
        "These may represent repeat admissions by the same patient. Confirm with the data custodian\n"
        "before proceeding; if confirmed, a patient-level (grouped) split is required instead of a\n"
        "random split, and the results below will be optimistically biased."
    )
    display(
        df_raw[df_raw.duplicated(subset=clinical, keep=False)]
        .sort_values(clinical)
        .head(20)
    )
else:
    print("No duplicated clinical records detected. Record-level independence is consistent with")
    print("the data, though it does not by itself confirm one record per unique patient.")

### 6.2 Completeness and storage types

The audit below distinguishes columns held as numeric types from those held as text. Several
laboratory variables are stored as text in the source file, which indicates the presence of
non-numeric characters — typically censoring notation, stray punctuation, or inconsistent decimal
separators introduced during manual entry. These are repaired in Section 7 with the number of
affected values reported, so that missingness arising from data repair can be distinguished from
missingness present in the source.

In [ ]:
quality = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_n": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2),
    "unique_n": df_raw.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)

quality["storage"] = np.where(
    quality["dtype"].eq("object"), "text", "numeric"
)

print("Table 1. Source data quality audit.")
display(save_table(quality, "table01_data_quality_audit"))

text_stored = quality.index[quality["storage"].eq("text")].tolist()
print(f"\nColumns stored as text: {text_stored}")
print("Of these, Sex, the Troponin assay type and UNIT are genuinely categorical;")
print("the remainder are numeric variables requiring repair.")

target_counts = df_raw["Heart Disease"].value_counts(dropna=False).sort_index()
target_table = pd.DataFrame({
    "n": target_counts,
    "percent": (target_counts / len(df_raw) * 100).round(2),
})
target_table.index = ["No heart disease (0)", "Heart disease (1)"][: len(target_table)]

print("\nOutcome distribution in the source file:")
display(target_table)

majority_share = target_counts.max() / target_counts.sum()
print(f"\nMajority-class share: {majority_share:.4f}")
print("Any classifier must exceed this accuracy to be informative. Class imbalance is mild,")
print("so ROC-AUC and F1 remain appropriate primary metrics without resampling.")

## 7. Variable Harmonisation

### 7.1 Column naming and numeric repair

Column names are normalised and the three text-stored laboratory variables are repaired. Three
distinct data-entry faults are present in the source: stray backtick characters, commas used as
decimal separators, and whitespace inserted before a decimal point. Each repair rule is applied
separately and its effect counted, so that the number of values recovered by repair and the number
that remained unparseable are both reported rather than silently absorbed into the missingness
statistics.

In [ ]:
df = df_raw.copy()
df.columns = df.columns.astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

df = df.rename(columns={
    "Troponin-I": "Troponin_I",
    "Troponin- I assay type": "Troponin_Assay_Type",
    "Heart Disease": "Heart_Disease",
})

EXPECTED_COLUMNS = {
    "SL", "Age", "Sex", "Height (cm)", "Weight (kg)", "BMI", "Family H/O",
    "Hypertension", "Diabetes", "Total_Cholesterol(mg/dL)", "BP(mmHg)",
    "H/O ChestPain", "RBS(mmol/L)", "HDL(mg/dL)", "LDL(mg/dL)",
    "Triglycerides(mg/dL)", "MaxHR", "Himoglobin", "Creatinine(mg/dL)",
    "Platelets", "Sodium(mmol/L)", "Potassium", "Chloride", "Troponin_I",
    "Troponin_Assay_Type", "Heart_Disease", "UNIT",
}
missing_expected = EXPECTED_COLUMNS - set(df.columns)
assert not missing_expected, f"Expected columns absent from source: {missing_expected}"

TEXT_NUMERIC = ["Himoglobin", "Potassium", "Chloride"]
repair_rows = []

for col in TEXT_NUMERIC:
    original = df[col].astype("string")
    present_before = original.notna().sum()
    parsed_before = pd.to_numeric(original, errors="coerce").notna().sum()

    repaired = (
        original
        .str.replace("`", "", regex=False)          # stray backtick
        .str.replace(",", ".", regex=False)          # comma decimal separator
        .str.replace(r"(?<=\d)\s+\.(?=\d)", ".", regex=True)  # space before decimal point
        .str.strip()
    )
    parsed_after = pd.to_numeric(repaired, errors="coerce").notna().sum()

    repair_rows.append({
        "Variable": col,
        "Non-empty in source": int(present_before),
        "Parsed without repair": int(parsed_before),
        "Recovered by repair": int(parsed_after - parsed_before),
        "Unparseable after repair": int(present_before - parsed_after),
    })
    df[col] = pd.to_numeric(repaired, errors="coerce")

repair_summary = pd.DataFrame(repair_rows).set_index("Variable")

print("Table 2. Numeric repair of text-stored laboratory variables.")
display(save_table(repair_summary, "table02_numeric_repair"))

unparseable_total = int(repair_summary["Unparseable after repair"].sum())
if unparseable_total:
    print(
        f"\n{unparseable_total} value(s) could not be parsed after repair and became missing.\n"
        "These are treated as missing at the imputation stage. Inspect them if the count is material."
    )
else:
    print("\nAll non-empty values were successfully parsed; no missingness was introduced by repair.")

### 7.2 Recomputation of body mass index

Body mass index is a deterministic function of height and weight, yet it is missing more often in
the source file than either of its constituents. Imputing a value that can be computed exactly
discards information for no benefit. BMI is therefore recomputed from height and weight wherever
both are recorded.

Two quantities are reported: the number of previously missing values recovered by computation, and
the largest discrepancy between supplied and computed values among records where both exist. The
second acts as an internal consistency check on the source file — a large discrepancy would indicate
that the supplied BMI was not derived from the supplied height and weight, which would itself
require investigation.

*(Change H5, approved.)*

In [ ]:
height_m = df["Height (cm)"] / 100.0
bmi_computed = df["Weight (kg)"] / (height_m ** 2)

both_available = df["Height (cm)"].notna() & df["Weight (kg)"].notna()
comparable = both_available & df["BMI"].notna()

discrepancy = (df.loc[comparable, "BMI"] - bmi_computed.loc[comparable]).abs()
recoverable = both_available & df["BMI"].isna()
n_recovered = int(recoverable.sum())

print("Body mass index consistency and recomputation")
print("-" * 60)
print(f"BMI missing in source                        : {int(df['BMI'].isna().sum())}")
print(f"Height and weight both available             : {int(both_available.sum())}")
print(f"Records where BMI is comparable              : {int(comparable.sum())}")
if comparable.any():
    print(f"Maximum |supplied - computed| discrepancy     : {discrepancy.max():.4f} kg/m2")
    print(f"Median |supplied - computed| discrepancy      : {discrepancy.median():.4f} kg/m2")
print(f"Missing BMI values recovered by computation  : {n_recovered}")
print("-" * 60)

df.loc[recoverable, "BMI"] = bmi_computed.loc[recoverable]
print(f"BMI missing after recomputation              : {int(df['BMI'].isna().sum())}")

if comparable.any() and discrepancy.max() > 0.5:
    print(
        "\nWARNING: supplied BMI values diverge from values computed from the supplied height and\n"
        "weight by more than 0.5 kg/m2. The three variables may not originate from the same\n"
        "measurement occasion. Raise this with the data custodian."
    )

## 8. Troponin-I Censoring and Assay Harmonisation

Troponin-I is recorded on two assay platforms whose reporting scales differ by three orders of
magnitude: a conventional assay reporting in ng/mL and a high-sensitivity assay reporting in ng/L.
Pooling these without conversion would treat a physiologically normal high-sensitivity result as an
extreme conventional result. High-sensitivity values are therefore divided by 1000 to place all
measurements on the ng/mL scale.

A subset of results is interval-censored rather than point-valued, reflecting the reporting limits
of the analyser:

| Notation | Interpretation | Handling |
|---|---|---|
| `>25000` | Above the upper reporting limit (right-censored) | Substituted at the limit; retained as an indicator variable |
| `<2.50` | Below the lower reporting limit (left-censored) | Substituted at LOD/√2 |
| `>2.5` | Ambiguous — inconsistent with either reporting limit | Treated as missing |

Left-censored values are substituted at LOD/√2 rather than at the limit itself. The true value lies
somewhere below the limit, so substituting the limit systematically biases the lower tail upward;
LOD/√2 is the standard convention in analytical chemistry for left-censored measurements and is less
biased than LOD/2 when the underlying distribution is approximately log-normal. Right-censored
values are retained at the limit, which is the conventional treatment, with the accompanying
indicator allowing any model to distinguish a censored observation from a genuine measurement at
that value.

*(Change H4, approved. The near-constant indicators for low censoring and ambiguous notation are
removed under change H7, approved: with three and two positive cases respectively they cannot
support estimation, and the ambiguous-notation indicator is in any case redundant with the
missingness indicator generated during imputation.)*

In [ ]:
troponin_text = df["Troponin_I"].astype("string").str.strip()

censored_high = troponin_text.str.fullmatch(r">\s*25000").fillna(False)
censored_low = troponin_text.str.fullmatch(r"<\s*2\.50").fillna(False)
ambiguous = troponin_text.str.fullmatch(r">\s*2\.5").fillna(False)

LOD_LOW = 2.50
LOD_HIGH = 25000.0
low_substitute = LOD_LOW / np.sqrt(2)

troponin_numeric = pd.to_numeric(
    troponin_text.str.replace(r"^[<>]\s*", "", regex=True), errors="coerce"
)
troponin_numeric.loc[censored_high] = LOD_HIGH
troponin_numeric.loc[censored_low] = low_substitute
troponin_numeric.loc[ambiguous] = np.nan

df["Troponin_I"] = troponin_numeric
df["Troponin_Censored_High"] = censored_high.astype(int)

HS_LABEL = "High-Sensitivity Troponin-I (ng/L)"
is_high_sensitivity = df["Troponin_Assay_Type"].eq(HS_LABEL)
df.loc[is_high_sensitivity & df["Troponin_I"].notna(), "Troponin_I"] /= 1000.0

print("Troponin-I harmonisation")
print("-" * 60)
print(f"Right-censored (>25000)                 : {int(censored_high.sum())}")
print(f"Left-censored (<2.50), set to LOD/sqrt2 : {int(censored_low.sum())}  ->  {low_substitute:.4f}")
print(f"Ambiguous (>2.5), set to missing        : {int(ambiguous.sum())}")
print(f"High-sensitivity records rescaled       : {int(is_high_sensitivity.sum())}")
print(f"Troponin-I missing after harmonisation  : {int(df['Troponin_I'].isna().sum())}")
print("-" * 60)
print("Distribution on the harmonised ng/mL scale:")
display(df["Troponin_I"].describe().to_frame("Troponin-I (ng/mL)").round(4))

print("\nNote: assay type is excluded from the predictor set (Section 9), but the rescaling above")
print("means assay provenance remains implicitly encoded in the harmonised values. This is")
print("unavoidable when pooling platforms and is recorded as a limitation.")

## 9. Modelling Population and Variable Exclusion

### 9.1 Population definition

Records for patients under 18 years of age are excluded. Paediatric cardiac disease differs
substantially from adult disease in aetiology, presentation and the interpretation of laboratory
reference ranges, and the number of such records is too small to support a separate model. The
resulting population is therefore adults admitted to the contributing facility.

### 9.2 Variable exclusion

Three variables are removed from the predictor set. Each exclusion is justified on its own terms
below.

**`SL`** is a record serial number. It carries no clinical meaning, but serial numbers frequently
correlate with the order of data collection, which may in turn correlate with ward, season or
referral pattern. Retaining it risks the model exploiting an artefact of file construction.

**`UNIT`** records the admitting ward, taking the values `CCU` (coronary care unit) and `General`.
Ward assignment is a consequence of clinical suspicion recorded at admission, and therefore reflects
a judgement already informed by the patient's presentation — the same judgement the model is
intended to support. Including it would allow the model to condition on the clinician's assessment
rather than on the underlying physiology, which constitutes leakage from the outcome-determination
process. The strength of the association is quantified below to document the basis for the decision.

**`Troponin_Assay_Type`** identifies the analyser platform. Assay selection is a property of the
laboratory workflow rather than of the patient, and where platform choice varies with clinical
urgency it encodes triage information. Its measurement content is preserved through the scale
harmonisation in Section 8.

In [ ]:
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

# Quantify the UNIT-outcome association that motivates its exclusion.
unit_table = pd.crosstab(df["UNIT"], df["Heart_Disease"])
chi2, p_unit, _, _ = st.chi2_contingency(unit_table)
unit_rates = pd.crosstab(df["UNIT"], df["Heart_Disease"], normalize="index").round(4)

print("Association between admitting ward and outcome (basis for excluding UNIT):")
display(unit_table)
print("Outcome rate within each ward:")
display(unit_rates)
print(f"Chi-square = {chi2:.2f}, p = {p_unit:.3e}")
print("A strong association confirms that ward assignment encodes clinical suspicion.")
print("-" * 70)

paediatric = df["Age"] < 18
print(f"\nPaediatric records excluded (age < 18): {int(paediatric.sum())}")
if paediatric.any():
    print(f"Ages excluded: {sorted(df.loc[paediatric, 'Age'].unique().tolist())}")

df_model = df.loc[~paediatric].copy()

EXCLUDED_VARIABLES = ["SL", "UNIT", "Troponin_Assay_Type"]
df_model = df_model.drop(columns=EXCLUDED_VARIABLES)

assert "Heart_Disease" in df_model.columns
assert not set(EXCLUDED_VARIABLES) & set(df_model.columns)

print(f"\nModelling population : {df_model.shape[0]} records")
print(f"Candidate predictors : {df_model.shape[1] - 1}")
display(
    df_model["Heart_Disease"].value_counts().sort_index()
    .rename(index={0: "No heart disease", 1: "Heart disease"})
    .to_frame("n")
)

### 9.3 Presentation labels

Variables are given clean display labels with units for use in all figures and tables. The source
file contains a misspelling of haemoglobin, which is corrected for presentation while the underlying
column name is left unchanged so that the code remains traceable to the source file. Labels marked
with a dagger carry units inferred from value ranges rather than documented in the source, and
require confirmation (see Section 3.4).

In [ ]:
DISPLAY_LABELS = {
    "Age": "Age (years)",
    "Sex": "Sex",
    "Height (cm)": "Height (cm)",
    "Weight (kg)": "Weight (kg)",
    "BMI": "Body mass index (kg/m2)",
    "Family H/O": "Family history of CVD",
    "Hypertension": "Hypertension",
    "Diabetes": "Diabetes mellitus",
    "Total_Cholesterol(mg/dL)": "Total cholesterol (mg/dL)",
    "BP(mmHg)": "Blood pressure (mmHg)",
    "H/O ChestPain": "History of chest pain",
    "RBS(mmol/L)": "Random blood sugar (mmol/L)",
    "HDL(mg/dL)": "HDL cholesterol (mg/dL)",
    "LDL(mg/dL)": "LDL cholesterol (mg/dL)",
    "Triglycerides(mg/dL)": "Triglycerides (mg/dL)",
    "MaxHR": "Maximum heart rate (bpm)",
    "Himoglobin": "Haemoglobin (g/dL)",
    "Creatinine(mg/dL)": "Creatinine (mg/dL)",
    "Platelets": "Platelet count (/uL)",
    "Sodium(mmol/L)": "Sodium (mmol/L)",
    "Potassium": "Potassium (mmol/L)",
    "Chloride": "Chloride (mmol/L)",
    "Troponin_I": "Troponin-I (ng/mL)",
    "Troponin_Censored_High": "Troponin-I above assay range",
    "Heart_Disease": "Heart disease",
}


def pretty(name):
    '''Map a raw column or transformed feature name to a presentation label.'''
    cleaned = str(name)
    for prefix in ("num__", "cat__", "remainder__"):
        if cleaned.startswith(prefix):
            cleaned = cleaned[len(prefix):]
    if "missingindicator" in cleaned:
        base = cleaned.split("missingindicator")[-1].strip("_")
        return f"{DISPLAY_LABELS.get(base, base)} [not recorded]"
    if cleaned in DISPLAY_LABELS:
        return DISPLAY_LABELS[cleaned]
    for raw, label in DISPLAY_LABELS.items():
        if cleaned.startswith(raw + "_"):
            return f"{label} = {cleaned[len(raw) + 1:]}"
    return cleaned


print("Example label mappings:")
for c in ["Himoglobin", "RBS(mmol/L)", "Family H/O", "num__LDL(mg/dL)", "cat__Sex_M"]:
    print(f"  {c:<28} ->  {pretty(c)}")

## 10. Derived-Variable Diagnostic: Maximum Heart Rate

Two features of `MaxHR` in the source data warrant investigation before it is used as a predictor.
It is completely observed, whereas every other laboratory and vital-sign variable has some
missingness; and it is very strongly associated with age. In a hospital dataset where even blood
pressure is incompletely recorded, a measured cardiac variable with no missing values is unusual.

The concern is that `MaxHR` may be age-predicted rather than measured. Age-predicted maximum heart
rate is calculated from formulae of the form `k − c × age`, and is a clinical convention rather than
an observation. If that is the case here, `MaxHR` contains no information beyond age, and any
apparent predictive contribution is an artefact of the transformation.

The diagnostic below fits a linear model of `MaxHR` on age and examines the residual scatter. If the
relationship is deterministic or near-deterministic, the coefficient of determination will approach
unity and the residuals will be negligible or discretised. Interpretation is reported in the cell
output, and the outcome should be confirmed against the data custodian's response to the question
raised in Section 3.4.

*(Change H2 is gated on this diagnostic and is not applied by default. Set `DROP_MAXHR = True` in
Section 4 only after reviewing the evidence below and confirming derivation with the data custodian.)*

In [ ]:
maxhr_data = df_model[["Age", "MaxHR"]].dropna()
slope, intercept, r_value, p_value, std_err = st.linregress(
    maxhr_data["Age"], maxhr_data["MaxHR"]
)
fitted = intercept + slope * maxhr_data["Age"]
residuals = maxhr_data["MaxHR"] - fitted

print("Diagnostic: is MaxHR measured or derived from age?")
print("-" * 68)
print(f"Pearson r                      : {r_value:.4f}")
print(f"R-squared                      : {r_value ** 2:.4f}")
print(f"Fitted relationship            : MaxHR = {intercept:.2f} + ({slope:.4f} x Age)")
print(f"Residual standard deviation    : {residuals.std():.4f} bpm")
print(f"Maximum absolute residual      : {residuals.abs().max():.4f} bpm")
print(f"Missing values in MaxHR        : {int(df_model['MaxHR'].isna().sum())}")
print(f"Missing values, other variables: "
      f"{int(df_model.drop(columns=['MaxHR']).isna().sum().sum())}")
print("-" * 68)

if r_value ** 2 > 0.95 and residuals.std() < 5:
    verdict = (
        "STRONG evidence of derivation. MaxHR is almost fully determined by age and carries\n"
        "essentially no independent information. Confirm with the data custodian and, if\n"
        "confirmed, set DROP_MAXHR = True."
    )
elif r_value ** 2 > 0.80:
    verdict = (
        "SUBSTANTIAL age dependence, exceeding what is typically observed for measured maximum\n"
        "heart rate (R-squared of roughly 0.09-0.16). Derivation is plausible. Confirm with the\n"
        "data custodian before relying on this variable."
    )
else:
    verdict = (
        "Age dependence is within the range expected for a measured variable. No action indicated\n"
        "on the basis of this diagnostic alone."
    )
print(verdict)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(maxhr_data["Age"], maxhr_data["MaxHR"], s=8, alpha=0.4)
axes[0].plot(maxhr_data["Age"], fitted, color="crimson", lw=1.5,
             label=f"OLS fit (R2 = {r_value ** 2:.3f})")
axes[0].set_xlabel("Age (years)")
axes[0].set_ylabel("Maximum heart rate (bpm)")
axes[0].set_title("(a) Maximum heart rate against age")
axes[0].legend()

axes[1].scatter(fitted, residuals, s=8, alpha=0.4)
axes[1].axhline(0, color="crimson", lw=1.2)
axes[1].set_xlabel("Fitted value (bpm)")
axes[1].set_ylabel("Residual (bpm)")
axes[1].set_title("(b) Residuals from the age model")

fig.suptitle("Figure 1. Diagnostic assessment of maximum heart rate as a derived variable", y=1.02)
plt.tight_layout()
save_figure("fig01_maxhr_derivation_diagnostic")
plt.show()

if DROP_MAXHR:
    df_model = df_model.drop(columns=["MaxHR"])
    print("\nDROP_MAXHR is enabled: MaxHR has been removed from the predictor set.")
else:
    print("\nDROP_MAXHR is disabled: MaxHR is retained pending confirmation.")

## 11. Physiological Plausibility Screening

Values are screened against wide bounds outside which a measurement is physiologically implausible
and most likely reflects a transcription or unit error. These are deliberately permissive
implausibility limits rather than clinical reference ranges: a haemoglobin of 6 g/dL is severely
abnormal but entirely possible, whereas a haemoglobin of 60 g/dL is not.

No values are altered at this stage. Automatic deletion of extreme observations in clinical data
risks removing exactly the severely ill patients the model most needs to characterise. The screen is
diagnostic: any flagged value should be verified against the source record before a decision is
taken, and the decision documented.

In [ ]:
PLAUSIBILITY_BOUNDS = {
    "Age": (18, 120),
    "Height (cm)": (100, 220),
    "Weight (kg)": (20, 250),
    "BMI": (10, 70),
    "Total_Cholesterol(mg/dL)": (50, 600),
    "BP(mmHg)": (40, 300),
    "RBS(mmol/L)": (1, 50),
    "HDL(mg/dL)": (5, 150),
    "LDL(mg/dL)": (10, 400),
    "Triglycerides(mg/dL)": (10, 2000),
    "MaxHR": (40, 230),
    "Himoglobin": (3, 22),
    "Creatinine(mg/dL)": (0.1, 20),
    "Platelets": (5_000, 1_500_000),
    "Sodium(mmol/L)": (100, 180),
    "Potassium": (1.5, 9.0),
    "Chloride": (70, 140),
    "Troponin_I": (0, 100),
}

plausibility_rows = []
for column, (low, high) in PLAUSIBILITY_BOUNDS.items():
    if column not in df_model.columns:
        continue
    values = df_model[column].dropna()
    out_of_range = ((values < low) | (values > high))
    plausibility_rows.append({
        "Variable": pretty(column),
        "Observed min": values.min(),
        "Observed max": values.max(),
        "Screen low": low,
        "Screen high": high,
        "Outside screen": int(out_of_range.sum()),
    })

plausibility = pd.DataFrame(plausibility_rows).set_index("Variable").round(3)

print("Table 3. Physiological plausibility screen (no values altered).")
display(save_table(plausibility, "table03_plausibility_screen"))

total_flagged = int(plausibility["Outside screen"].sum())
if total_flagged:
    print(f"\n{total_flagged} value(s) fall outside the plausibility screen and require verification")
    print("against the source records before submission. Flagged variables:")
    display(plausibility.loc[plausibility["Outside screen"] > 0, ["Observed min", "Observed max",
                                                                 "Screen low", "Screen high",
                                                                 "Outside screen"]])
else:
    print("\nAll observed values fall within physiologically plausible bounds.")

## 12. Exploratory Data Analysis

Exploratory analysis serves two purposes here: to characterise the cohort for the reader, and to
document the empirical basis for later modelling decisions. It is reported on the full modelling
population, which is appropriate for descriptive purposes. No variable is selected, removed or
transformed on the basis of anything in this section; feature screening that could influence the
model is confined to the training partition in Section 14.

### 12.1 Cohort characteristics by outcome

Table 4 reports the distribution of every candidate predictor stratified by outcome status.
Continuous variables are summarised as median with interquartile range and compared using the
Mann-Whitney U test; binary and categorical variables are summarised as counts with percentages and
compared using the chi-square test.

The p-values are descriptive. Twenty-six comparisons are made without correction for multiple
testing, and these tests do not inform any modelling decision, so they should be read as a
characterisation of the cohort rather than as inferential findings.

In [ ]:
BINARY_VARS = [
    c for c in ["Family H/O", "Hypertension", "Diabetes", "H/O ChestPain",
                "Troponin_Censored_High"]
    if c in df_model.columns
]
CATEGORICAL_VARS = [c for c in df_model.columns
                    if df_model[c].dtype == object and c != "Heart_Disease"]
CONTINUOUS_VARS = [
    c for c in df_model.columns
    if c not in BINARY_VARS + CATEGORICAL_VARS + ["Heart_Disease"]
    and pd.api.types.is_numeric_dtype(df_model[c])
]

group_0 = df_model[df_model["Heart_Disease"] == 0]
group_1 = df_model[df_model["Heart_Disease"] == 1]
table1_rows = []


def _iqr_summary(series):
    series = series.dropna()
    if series.empty:
        return "-"
    return f"{series.median():.1f} [{series.quantile(0.25):.1f}-{series.quantile(0.75):.1f}]"


for column in CONTINUOUS_VARS:
    a, b = group_0[column].dropna(), group_1[column].dropna()
    p = st.mannwhitneyu(a, b, alternative="two-sided").pvalue if len(a) > 1 and len(b) > 1 else np.nan
    table1_rows.append({
        "Variable": pretty(column),
        "Summary": "median [IQR]",
        "No heart disease": _iqr_summary(group_0[column]),
        "Heart disease": _iqr_summary(group_1[column]),
        "Missing n": int(df_model[column].isna().sum()),
        "p-value": p,
    })

for column in BINARY_VARS + CATEGORICAL_VARS:
    if column in CATEGORICAL_VARS:
        level = sorted(df_model[column].dropna().unique())[-1]
        label = f"{pretty(column)} = {level}"
        mask_0 = (group_0[column] == level).sum()
        mask_1 = (group_1[column] == level).sum()
        denom_0 = group_0[column].notna().sum()
        denom_1 = group_1[column].notna().sum()
        contingency = pd.crosstab(df_model[column], df_model["Heart_Disease"])
    else:
        label = f"{pretty(column)} = 1"
        mask_0 = (group_0[column] == 1).sum()
        mask_1 = (group_1[column] == 1).sum()
        denom_0 = group_0[column].notna().sum()
        denom_1 = group_1[column].notna().sum()
        contingency = pd.crosstab(df_model[column], df_model["Heart_Disease"])

    p = st.chi2_contingency(contingency).pvalue if contingency.shape[0] > 1 else np.nan
    table1_rows.append({
        "Variable": label,
        "Summary": "n (%)",
        "No heart disease": f"{mask_0} ({100 * mask_0 / max(denom_0, 1):.1f}%)",
        "Heart disease": f"{mask_1} ({100 * mask_1 / max(denom_1, 1):.1f}%)",
        "Missing n": int(df_model[column].isna().sum()),
        "p-value": p,
    })

table1 = pd.DataFrame(table1_rows).set_index("Variable")
table1["p-value"] = table1["p-value"].apply(
    lambda v: "<0.001" if pd.notna(v) and v < 0.001 else (f"{v:.3f}" if pd.notna(v) else "-")
)

print(f"Table 4. Cohort characteristics by outcome status "
      f"(n = {len(df_model)}; no heart disease = {len(group_0)}, heart disease = {len(group_1)}).")
display(save_table(table1, "table04_cohort_characteristics"))

### 12.2 Outcome balance and data completeness

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5),
                         gridspec_kw={"width_ratios": [1, 1.6]})

counts = df_model["Heart_Disease"].value_counts().sort_index()
bars = axes[0].bar(["No heart disease", "Heart disease"], counts.values,
                   color=["#4C72B0", "#C44E52"], width=0.6)
axes[0].set_ylabel("Number of records")
axes[0].set_title("(a) Outcome distribution")
axes[0].set_ylim(0, counts.max() * 1.15)
for bar, value in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, value + counts.max() * 0.02,
                 f"{value}\n({100 * value / counts.sum():.1f}%)", ha="center", va="bottom")

missing = (
    df_model.isna().sum().to_frame("n")
    .assign(percent=lambda t: 100 * t["n"] / len(df_model))
    .query("n > 0").sort_values("percent")
)
if not missing.empty:
    axes[1].barh([pretty(i) for i in missing.index], missing["percent"], color="#55A868")
    axes[1].set_xlabel("Missing values (% of records)")
    axes[1].set_title("(b) Completeness by variable")
    for y, (value, count) in enumerate(zip(missing["percent"], missing["n"])):
        axes[1].text(value + 0.1, y, f"{count}", va="center", fontsize=8)
else:
    axes[1].text(0.5, 0.5, "No missing values", ha="center", va="center")
    axes[1].set_axis_off()

fig.suptitle("Figure 2. Outcome balance and data completeness in the modelling population", y=1.01)
plt.tight_layout()
save_figure("fig02_outcome_and_completeness")
plt.show()

print(f"Outcome ratio (disease : no disease) = {counts[1] / counts[0]:.2f} : 1")
print("Imbalance is mild. Class weighting is applied within the estimators rather than resampling,")
print("which avoids generating synthetic clinical records.")

### 12.3 Distribution of the most strongly associated predictors

The panels below show the distribution of the six continuous predictors most strongly associated
with the outcome, stratified by outcome status.

These distributions carry diagnostic weight beyond description. Routine laboratory measurements in
real clinical cohorts overlap substantially between diseased and non-diseased patients; a single
lipid or glucose value does not ordinarily separate the two groups. If the panels below show clean
separation with little overlap, that is not a favourable finding — it indicates that the outcome
label is closely tied to these measurements, which points either to the label having been assigned
using them, or to the data being partly synthetic. Either possibility must be resolved through
Section 3.3 before the model's performance can be interpreted.

In [ ]:
association = (
    df_model[CONTINUOUS_VARS + ["Heart_Disease"]]
    .corr(numeric_only=True)["Heart_Disease"].drop("Heart_Disease")
    .abs().sort_values(ascending=False)
)
top_six = association.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, column in zip(axes.ravel(), top_six):
    subset = df_model[[column, "Heart_Disease"]].dropna()
    sns.violinplot(data=subset, x="Heart_Disease", y=column, ax=ax,
                   hue="Heart_Disease", legend=False,
                   palette=["#4C72B0", "#C44E52"], inner="quartile", cut=0)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["No disease", "Disease"])
    ax.set_xlabel("")
    ax.set_ylabel(pretty(column))
    ax.set_title(f"|r| = {association[column]:.3f}", fontsize=10)

fig.suptitle("Figure 3. Distribution of the six most strongly associated continuous predictors, "
             "by outcome status", y=1.01)
plt.tight_layout()
save_figure("fig03_top_predictor_distributions")
plt.show()

print("Absolute point-biserial correlation with the outcome, all continuous predictors:")
display(association.to_frame("|r|").round(4).rename(index=pretty))

extreme = association[association > 0.5]
if len(extreme) > 0:
    print(
        f"\nNOTE: {len(extreme)} continuous predictor(s) show |r| > 0.5 with the outcome.\n"
        "Associations of this magnitude between a single routine laboratory measurement and a\n"
        "clinical diagnosis are substantially larger than published cohort studies report.\n"
        "This is the principal evidence motivating the outcome-definition question in Section 3.3\n"
        "and must be addressed in the limitations."
    )

## 13. Data Partitioning

The data are partitioned once, before any model fitting or feature screening, into a training set
(80%) and a held-out test set (20%), stratified on the outcome to preserve class proportions.

The test partition is used exclusively for final reporting in Sections 20 to 24. It plays no part in
imputation, scaling, hyperparameter search, model comparison or model selection. Every decision that
could be influenced by observed performance is taken using the training partition alone, via the
nested cross-validation procedure in Section 17.

In [ ]:
y = df_model["Heart_Disease"].astype(int)
X = df_model.drop(columns=["Heart_Disease"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "Training n": y_train.value_counts().sort_index(),
    "Training %": (y_train.value_counts(normalize=True).sort_index() * 100).round(1),
    "Test n": y_test.value_counts().sort_index(),
    "Test %": (y_test.value_counts(normalize=True).sort_index() * 100).round(1),
})
split_summary.index = ["No heart disease (0)", "Heart disease (1)"]

print(f"Training records : {len(X_train)}")
print(f"Test records     : {len(X_test)}")
print(f"Predictors       : {X.shape[1]}")
print(f"Events per variable (training) : {int(y_train.sum()) / X.shape[1]:.1f}")
print("An events-per-variable ratio above 10 is conventionally regarded as adequate for")
print("logistic regression without penalisation-specific sample size adjustment.\n")
display(save_table(split_summary, "table05_partition_summary"))

## 14. Correlation and Redundancy Screening

Correlation structure is examined on the training partition only, so that nothing observed here can
transmit information from the test set into a modelling decision. Full-cohort values are reported
alongside for comparison; any material divergence between the two would indicate an unrepresentative
partition.

Strongly correlated predictor pairs are flagged but not removed automatically. Collinearity inflates
the variance of logistic regression coefficients and makes coefficient-based interpretation
unreliable, but it does not degrade the discriminative performance of tree ensembles. The
appropriate response therefore depends on which model is selected, and the flagged pairs are carried
forward into the interpretation in Section 25.

In [ ]:
numeric_columns = X_train.select_dtypes(include=np.number).columns.tolist()

train_frame = X_train[numeric_columns].join(y_train.rename("Heart_Disease"))
corr_train = train_frame.corr(numeric_only=True)
corr_full = df_model[numeric_columns + ["Heart_Disease"]].corr(numeric_only=True)

comparison_corr = pd.DataFrame({
    "|r| training partition": corr_train["Heart_Disease"].drop("Heart_Disease").abs(),
    "|r| full cohort": corr_full["Heart_Disease"].drop("Heart_Disease").abs(),
}).sort_values("|r| training partition", ascending=False)
comparison_corr["Difference"] = (
    comparison_corr["|r| training partition"] - comparison_corr["|r| full cohort"]
).abs()
comparison_corr.index = [pretty(i) for i in comparison_corr.index]

print("Table 6. Association with the outcome: training partition versus full cohort.")
display(save_table(comparison_corr.round(4), "table06_outcome_association"))
print(f"Maximum divergence between partitions: {comparison_corr['Difference'].max():.4f}")

predictor_corr = corr_train.drop(index="Heart_Disease").drop(columns="Heart_Disease").abs()
mask = np.triu(np.ones(predictor_corr.shape, dtype=bool))

ordered = predictor_corr.copy()
ordered.index = [pretty(i) for i in ordered.index]
ordered.columns = [pretty(c) for c in ordered.columns]

plt.figure(figsize=(11, 9))
sns.heatmap(ordered, mask=mask, cmap="rocket_r", vmin=0, vmax=1,
            square=True, linewidths=0.4, cbar_kws={"label": "|Pearson r|", "shrink": 0.7})
plt.title("Figure 4. Absolute pairwise correlation between numeric predictors (training partition)")
plt.tight_layout()
save_figure("fig04_predictor_correlation")
plt.show()

COLLINEARITY_THRESHOLD = 0.85
flagged = (
    predictor_corr.where(np.triu(np.ones(predictor_corr.shape), k=1).astype(bool))
    .stack().reset_index()
    .rename(columns={"level_0": "Variable 1", "level_1": "Variable 2", 0: "|r|"})
    .query("`|r|` >= @COLLINEARITY_THRESHOLD")
    .sort_values("|r|", ascending=False)
)
flagged["Variable 1"] = flagged["Variable 1"].map(pretty)
flagged["Variable 2"] = flagged["Variable 2"].map(pretty)

print(f"\nPredictor pairs with |r| >= {COLLINEARITY_THRESHOLD}:")
if flagged.empty:
    print("None.")
else:
    display(flagged.reset_index(drop=True).round(4))
    print("These pairs are retained. Their presence is carried into the interpretation of")
    print("coefficient- and importance-based explanations in Section 25.")

## 15. Preprocessing and the Informative-Missingness Diagnostic

### 15.1 Preprocessing specification

All preprocessing is defined as a `Pipeline` and fitted inside every cross-validation fold, so that
imputation medians, scaling parameters and category encodings are estimated only from the data
available to the model at that point in the procedure. No statistic computed from the test set, or
from a validation fold, ever influences a transformation applied to it.

| Step | Numeric variables | Categorical variables |
|---|---|---|
| Imputation | Median | Most frequent category |
| Missingness indicator | Optional (see 15.2) | Not applied |
| Encoding | — | One-hot, unknown categories ignored |
| Scaling | Standardisation, for distance- and penalty-based models only | Not applied |

Standardisation is applied for logistic regression and the support vector machine, whose objective
functions are scale-sensitive, and omitted for the decision tree and random forest, which are
invariant to monotone transformations of individual predictors.

### 15.2 Informative missingness

Missing-value indicators are generated by default, on the reasoning that the absence of a
measurement can itself be informative. In hospital data this reasoning cuts both ways. A troponin
assay is ordered when myocardial injury is suspected; a lipid panel is ordered under different
circumstances. An indicator for whether a test was performed may therefore encode which patients the
attending clinician was already concerned about, and a model exploiting that signal is conditioning
on the clinical judgement it was intended to support rather than on the patient's physiology.

The diagnostic below tests this directly by fitting a logistic regression on the missingness
indicators alone, with no measured values, evaluated by cross-validation on the training partition.
A model built from nothing but the pattern of which tests were ordered should perform close to
chance. Performance materially above chance indicates that missingness carries outcome information.

*(Change H3 is gated on this diagnostic and is not applied by default. Set
`USE_MISSINGNESS_INDICATORS = False` in Section 4 if the diagnostic indicates leakage.)*

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

try:
    ONE_HOT = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # scikit-learn < 1.2
    ONE_HOT = OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(scale_numeric=True, add_indicator=None):
    '''Construct a leakage-safe ColumnTransformer for the current feature set.'''
    if add_indicator is None:
        add_indicator = USE_MISSINGNESS_INDICATORS

    numeric_steps = [("impute", SimpleImputer(strategy="median", add_indicator=add_indicator))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))

    categorical_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", ONE_HOT),
    ])

    return ColumnTransformer(
        [("num", Pipeline(numeric_steps), numeric_features),
         ("cat", categorical_pipeline, categorical_features)],
        remainder="drop",
        verbose_feature_names_out=True,
    )


print(f"Numeric predictors     : {len(numeric_features)}")
print(f"Categorical predictors : {categorical_features}")

incomplete = [c for c in numeric_features if X_train[c].isna().any()]
print(f"Numeric predictors with missing values in training data : {len(incomplete)}")
print(f"Missingness indicators enabled : {USE_MISSINGNESS_INDICATORS}")

probe = build_preprocessor(scale_numeric=True).fit(X_train)
print(f"Transformed feature space : {len(probe.get_feature_names_out())} columns")

In [ ]:
# Diagnostic: can the outcome be predicted from the pattern of missingness alone?
outer_cv = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)

if incomplete:
    missingness_matrix = X_train[incomplete].isna().astype(int)

    indicator_model = Pipeline([
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced",
                                     random_state=RANDOM_STATE))
    ])

    fold_auc = []
    for train_idx, valid_idx in outer_cv.split(missingness_matrix, y_train):
        indicator_model.fit(missingness_matrix.iloc[train_idx], y_train.iloc[train_idx])
        scores = indicator_model.predict_proba(missingness_matrix.iloc[valid_idx])[:, 1]
        fold_auc.append(roc_auc_score(y_train.iloc[valid_idx], scores))

    indicator_auc = float(np.mean(fold_auc))
    indicator_sd = float(np.std(fold_auc))

    print("Diagnostic: outcome prediction from missingness pattern alone")
    print("-" * 68)
    print(f"Indicator variables used     : {len(incomplete)}")
    print(f"Cross-validated ROC-AUC      : {indicator_auc:.4f} (SD {indicator_sd:.4f})")
    print(f"Chance performance           : 0.5000")
    print("-" * 68)

    if indicator_auc > 0.65:
        print(
            "CONCERN: the pattern of which tests were performed predicts the outcome well above\n"
            "chance. Missingness is not at random and plausibly encodes clinical suspicion.\n"
            "Recommended action: set USE_MISSINGNESS_INDICATORS = False in Section 4, re-run, and\n"
            "report the difference in performance as a sensitivity analysis."
        )
    elif indicator_auc > 0.58:
        print(
            "BORDERLINE: missingness carries some outcome information. Retaining the indicators is\n"
            "defensible, but the diagnostic value above must be reported in the limitations."
        )
    else:
        print(
            "Missingness carries little outcome information. Retaining the indicators is unlikely\n"
            "to introduce leakage from the clinical decision process."
        )
else:
    indicator_auc, indicator_sd = float("nan"), float("nan")
    print("No missing values in the training partition; diagnostic not applicable.")

## 16. Candidate Models and Hyperparameter Search Space

Four classifiers spanning distinct inductive biases are evaluated: a penalised linear model, a
single axis-aligned decision tree, a maximum-margin kernel method, and a bagged tree ensemble. All
apply balanced class weighting, which reweights the loss in inverse proportion to class frequency
and avoids generating synthetic patient records.

Hyperparameter grids are specified below and are held fixed for both the inner tuning loop of the
nested procedure and the final fit on the full training partition. ROC-AUC is the tuning criterion,
being threshold-independent and appropriate under mild class imbalance.

In [ ]:
def model_specifications():
    '''Return the candidate pipelines and their hyperparameter grids.'''
    return {
        "Logistic Regression": (
            Pipeline([
                ("preprocess", build_preprocessor(scale_numeric=True)),
                ("model", LogisticRegression(max_iter=5000, class_weight="balanced",
                                             random_state=RANDOM_STATE)),
            ]),
            {
                "model__C": [0.01, 0.1, 1, 10],
                "model__penalty": ["l1", "l2"],
                "model__solver": ["liblinear"],
            },
        ),
        "Decision Tree": (
            Pipeline([
                ("preprocess", build_preprocessor(scale_numeric=False)),
                ("model", DecisionTreeClassifier(class_weight="balanced",
                                                 random_state=RANDOM_STATE)),
            ]),
            {
                "model__max_depth": [3, 5, 8, None],
                "model__min_samples_split": [2, 10],
                "model__min_samples_leaf": [1, 5, 10],
            },
        ),
        "Support Vector Machine": (
            Pipeline([
                ("preprocess", build_preprocessor(scale_numeric=True)),
                ("model", SVC(probability=True, class_weight="balanced",
                              random_state=RANDOM_STATE)),
            ]),
            {
                "model__C": [0.1, 1, 10],
                "model__kernel": ["linear", "rbf"],
                "model__gamma": ["scale", "auto"],
            },
        ),
        "Random Forest": (
            Pipeline([
                ("preprocess", build_preprocessor(scale_numeric=False)),
                ("model", RandomForestClassifier(class_weight="balanced", n_jobs=-1,
                                                 random_state=RANDOM_STATE)),
            ]),
            {
                "model__n_estimators": [200, 400],
                "model__max_depth": [None, 5, 10],
                "model__min_samples_leaf": [1, 5],
                "model__max_features": ["sqrt", "log2"],
            },
        ),
    }


# Parsimony ordering used by the pre-specified selection rule (lower is simpler).
PARSIMONY_RANK = {
    "Logistic Regression": 1,
    "Decision Tree": 2,
    "Support Vector Machine": 3,
    "Random Forest": 4,
}

grid_sizes = {
    name: int(np.prod([len(v) for v in grid.values()]))
    for name, (_, grid) in model_specifications().items()
}
print("Hyperparameter configurations per model:")
for name, size in grid_sizes.items():
    print(f"  {name:<26} {size:>3} configurations")
print(f"\nTotal fits in the nested procedure: "
      f"{sum(grid_sizes.values()) * N_INNER_FOLDS * N_OUTER_FOLDS:,} "
      f"(plus {sum(grid_sizes.values()) * N_INNER_FOLDS:,} for the final tuning pass)")

## 17. Nested Cross-Validation

### 17.1 Rationale

When hyperparameters are chosen by cross-validation and the resulting cross-validated score is then
reported as an estimate of generalisation performance, that score is optimistically biased: the
configuration was selected because it performed well on precisely the folds used to evaluate it. The
magnitude of the bias grows with the size of the search space, and it is not removed by re-scoring
the selected configuration on the same folds.

Nested cross-validation separates the two roles. An inner loop selects hyperparameters using only
the data in each outer training fold; the outer loop evaluates the entire selection procedure on
data that played no part in it. The resulting estimate is of the performance of the *method* — tuning
included — rather than of one configuration chosen with knowledge of the evaluation folds.

### 17.2 Procedure

For each of five stratified outer folds, a five-fold grid search is conducted on the outer training
portion and the selected configuration is applied to the held-out outer fold. Predicted
probabilities from the outer folds are retained, giving one out-of-fold prediction for every
training record. These pooled predictions provide both an unbiased performance estimate and the
basis for the statistical model comparison in Section 18.

*(Change H1, approved. Note that the estimates produced here are expected to be lower than
conventional cross-validated scores. That difference is the bias being removed, not a deterioration
in the model.)*

> **Runtime.** This section is the computationally expensive part of the notebook. Expect several
> minutes on a standard Colab CPU runtime.

In [ ]:
inner_cv = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)

nested_oof_proba = {}
nested_fold_scores = {}
nested_selected_params = {}

X_train_reset = X_train.reset_index(drop=True)
y_train_reset = y_train.reset_index(drop=True)

for name, (pipeline, grid) in model_specifications().items():
    oof = np.full(len(X_train_reset), np.nan)
    fold_scores, fold_params = [], []

    for fold_index, (fit_idx, eval_idx) in enumerate(
        outer_cv.split(X_train_reset, y_train_reset), start=1
    ):
        search = GridSearchCV(pipeline, grid, scoring="roc_auc",
                              cv=inner_cv, n_jobs=-1, refit=True)
        search.fit(X_train_reset.iloc[fit_idx], y_train_reset.iloc[fit_idx])

        proba = search.best_estimator_.predict_proba(X_train_reset.iloc[eval_idx])[:, 1]
        oof[eval_idx] = proba

        fold_scores.append(roc_auc_score(y_train_reset.iloc[eval_idx], proba))
        fold_params.append(search.best_params_)

    nested_oof_proba[name] = oof
    nested_fold_scores[name] = np.array(fold_scores)
    nested_selected_params[name] = fold_params
    print(f"{name:<26} nested ROC-AUC = {np.mean(fold_scores):.4f} "
          f"(SD {np.std(fold_scores):.4f})")

assert all(not np.isnan(v).any() for v in nested_oof_proba.values())
print("\nOut-of-fold predictions complete for all models.")

In [ ]:
nested_rows = []
for name, scores in nested_fold_scores.items():
    pooled = roc_auc_score(y_train_reset, nested_oof_proba[name])
    labels = (nested_oof_proba[name] >= 0.5).astype(int)
    nested_rows.append({
        "Model": name,
        "Nested ROC-AUC (mean)": scores.mean(),
        "Nested ROC-AUC (SD)": scores.std(),
        "Nested ROC-AUC (pooled)": pooled,
        "Nested accuracy": accuracy_score(y_train_reset, labels),
        "Nested sensitivity": recall_score(y_train_reset, labels, zero_division=0),
        "Nested specificity": recall_score(y_train_reset, labels, pos_label=0, zero_division=0),
        "Nested F1": f1_score(y_train_reset, labels, zero_division=0),
    })

nested_summary = (
    pd.DataFrame(nested_rows)
    .sort_values("Nested ROC-AUC (pooled)", ascending=False)
    .reset_index(drop=True).set_index("Model")
)

print("Table 7. Nested cross-validation performance (training partition only).")
print("These are unbiased estimates of the full modelling procedure including hyperparameter search.")
display(save_table(nested_summary.round(4), "table07_nested_cv_performance"))

print("\nHyperparameter configurations selected across outer folds:")
for name, params in nested_selected_params.items():
    distinct = {json.dumps(p, sort_keys=True) for p in params}
    print(f"\n  {name} — {len(distinct)} distinct configuration(s) across {len(params)} folds")
    for entry in sorted(distinct):
        occurrences = sum(json.dumps(p, sort_keys=True) == entry for p in params)
        print(f"    [{occurrences}/{len(params)}] {entry}")
print("\nConfigurations that vary substantially across folds indicate that the search is responding")
print("to fold-level noise, which is itself an argument for the more parsimonious model.")

## 18. Statistical Comparison of Candidate Models

Differences in ROC-AUC between models must be assessed against sampling variability before any
model is described as superior. The DeLong test provides an asymptotically exact comparison of two
correlated ROC curves — correlated because both models are evaluated on the same observations — using
the theory of generalised U-statistics. The implementation below follows the fast algorithm of Sun
and Xu (2014), which computes the covariance structure in O(n log n) using midranks.

The comparison is performed on the pooled out-of-fold predictions from the nested procedure, not on
the test set. Using the test set to choose between models would compromise its status as an
untouched estimate of generalisation performance. The test set is used in Section 20 only to
corroborate a decision already taken.

In [ ]:
def _midrank(x):
    '''Midranks of x, handling ties (Sun & Xu, 2014).'''
    order = np.argsort(x)
    sorted_x = x[order]
    n = len(x)
    ranks_sorted = np.zeros(n, dtype=float)
    i = 0
    while i < n:
        j = i
        while j < n and sorted_x[j] == sorted_x[i]:
            j += 1
        ranks_sorted[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    ranks = np.empty(n, dtype=float)
    ranks[order] = ranks_sorted
    return ranks


def _fast_delong(predictions_sorted, n_positive):
    '''Return AUCs and their covariance matrix for k models on shared observations.'''
    m, n = n_positive, predictions_sorted.shape[1] - n_positive
    positives = predictions_sorted[:, :m]
    negatives = predictions_sorted[:, m:]
    k = predictions_sorted.shape[0]

    tx = np.empty((k, m)); ty = np.empty((k, n)); tz = np.empty((k, m + n))
    for r in range(k):
        tx[r] = _midrank(positives[r])
        ty[r] = _midrank(negatives[r])
        tz[r] = _midrank(predictions_sorted[r])

    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1) / (2 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1 - (tz[:, m:] - ty) / m
    sx = np.cov(v01, ddof=1).reshape(k, k)
    sy = np.cov(v10, ddof=1).reshape(k, k)
    return aucs, sx / m + sy / n


def delong_test(y_true, proba_a, proba_b):
    '''Two-sided DeLong test for equality of two correlated ROC-AUCs.'''
    y_true = np.asarray(y_true).astype(int)
    order = np.argsort(-y_true, kind="mergesort")   # positives first
    n_positive = int(y_true.sum())

    stacked = np.vstack([np.asarray(proba_a, dtype=float),
                         np.asarray(proba_b, dtype=float)])[:, order]
    aucs, covariance = _fast_delong(stacked, n_positive)

    contrast = np.array([1.0, -1.0])
    variance = float(contrast @ covariance @ contrast)
    difference = float(aucs[0] - aucs[1])

    if variance <= 0:
        return {"auc_a": float(aucs[0]), "auc_b": float(aucs[1]),
                "difference": difference, "z": np.nan, "p_value": np.nan,
                "ci_low": np.nan, "ci_high": np.nan}

    se = np.sqrt(variance)
    z = difference / se
    p = 2 * st.norm.sf(abs(z))
    critical = st.norm.ppf(1 - ALPHA / 2)
    return {"auc_a": float(aucs[0]), "auc_b": float(aucs[1]), "difference": difference,
            "z": float(z), "p_value": float(p),
            "ci_low": difference - critical * se, "ci_high": difference + critical * se}


# Verify the implementation reproduces sklearn's AUC before relying on it.
_check_model = list(nested_oof_proba)[0]
_check = delong_test(y_train_reset, nested_oof_proba[_check_model], nested_oof_proba[_check_model])
assert abs(_check["auc_a"] - roc_auc_score(y_train_reset, nested_oof_proba[_check_model])) < 1e-8, \
    "DeLong AUC does not match scikit-learn; implementation error."
print("DeLong implementation verified against scikit-learn's ROC-AUC.")

In [ ]:
reference_model = nested_summary.index[0]
print(f"Reference model (highest nested ROC-AUC): {reference_model}\n")

comparison_rows = []
for name in nested_summary.index:
    if name == reference_model:
        continue
    result = delong_test(y_train_reset,
                         nested_oof_proba[reference_model],
                         nested_oof_proba[name])
    comparison_rows.append({
        "Comparison": f"{reference_model} vs {name}",
        "AUC difference": result["difference"],
        f"{int((1 - ALPHA) * 100)}% CI": f"[{result['ci_low']:.4f}, {result['ci_high']:.4f}]",
        "z": result["z"],
        "p-value": result["p_value"],
        "Distinguishable": "Yes" if (pd.notna(result["p_value"]) and result["p_value"] < ALPHA) else "No",
    })

delong_table = pd.DataFrame(comparison_rows).set_index("Comparison")

print("Table 8. DeLong tests against the leading model (nested out-of-fold predictions).")
display(save_table(delong_table.round(4), "table08_delong_comparisons"))

equivalent_models = [reference_model] + [
    row["Comparison"].split(" vs ")[1]
    for _, row in delong_table.reset_index().iterrows()
    if row["Distinguishable"] == "No"
]
print(f"\nModels statistically indistinguishable from {reference_model} at alpha = {ALPHA}:")
for name in equivalent_models:
    print(f"  - {name}")

## 19. Model Selection

The pre-specified rule from Section 2.3 is applied. The model with the highest nested
cross-validation AUC forms the reference; every model whose AUC is not statistically distinguishable
from it constitutes the equivalence set; and the most parsimonious member of that set is selected.

The justification for preferring parsimony under equivalence is threefold. A model that cannot be
shown to perform better does not warrant the additional complexity. Simpler models are less prone to
the variance that produces unstable behaviour on new data from a different site. And in the
deployment context of the parent project — a web application intended for settings with limited
computational infrastructure — a linear model with reportable coefficients is materially easier to
serve, audit and explain to a clinical reviewer than a several-hundred-tree ensemble.

The selected model is then refitted on the complete training partition using the same hyperparameter
grid and inner cross-validation scheme.

In [ ]:
selection_frame = nested_summary.copy()
selection_frame["Parsimony rank"] = [PARSIMONY_RANK[name] for name in selection_frame.index]
selection_frame["In equivalence set"] = [
    name in equivalent_models for name in selection_frame.index
]

SELECTED_MODEL = min(equivalent_models, key=lambda name: PARSIMONY_RANK[name])

print("Table 9. Model selection under the pre-specified rule.")
display(save_table(
    selection_frame[["Nested ROC-AUC (pooled)", "Nested ROC-AUC (SD)", "Nested sensitivity",
                     "Nested specificity", "Nested F1", "Parsimony rank", "In equivalence set"]]
    .round(4),
    "table09_model_selection"
))

print(f"\nReference model (highest nested AUC) : {reference_model}")
print(f"Equivalence set                      : {', '.join(equivalent_models)}")
print(f"SELECTED MODEL                       : {SELECTED_MODEL}")
if SELECTED_MODEL != reference_model:
    print(
        f"\n{SELECTED_MODEL} is selected in preference to {reference_model} because the DeLong test\n"
        f"provides no evidence that they differ in discriminative performance, and it is the more\n"
        f"parsimonious of the two under the ordering fixed in Section 2.3."
    )
else:
    print(f"\n{SELECTED_MODEL} is both the leading and the most parsimonious model in the "
          f"equivalence set.")

In [ ]:
# Refit every candidate on the full training partition for reporting; the selected model is
# carried forward. Refitting occurs after selection and cannot influence it.
fitted_searches = {}
for name, (pipeline, grid) in model_specifications().items():
    search = GridSearchCV(pipeline, grid, scoring="roc_auc",
                          cv=inner_cv, n_jobs=-1, refit=True)
    search.fit(X_train, y_train)
    fitted_searches[name] = search
    print(f"{name:<26} tuned on full training partition")

FINAL_PIPELINE = fitted_searches[SELECTED_MODEL].best_estimator_
FINAL_PARAMS = fitted_searches[SELECTED_MODEL].best_params_

print(f"\nFinal model      : {SELECTED_MODEL}")
print(f"Hyperparameters  : {json.dumps(FINAL_PARAMS, indent=2, default=str)}")

## 20. Held-Out Test Performance

The test partition is evaluated here for the first time. Because selection is already complete,
these figures are an honest estimate of the selected model's performance on data drawn from the same
distribution.

Metrics conventional in clinical prediction research are reported: sensitivity (the proportion of
patients with heart disease correctly identified), specificity (the proportion without heart disease
correctly excluded), positive and negative predictive values, and balanced accuracy. Predictive
values depend on outcome prevalence and therefore apply to this cohort's case mix; they would differ
in a screening population with lower prevalence.

Every metric is accompanied by a 95% percentile bootstrap confidence interval computed over 2,000
stratified resamples of the test partition. With 207 test records, point estimates alone convey a
precision the sample size does not support.

A majority-class baseline is included for reference. It represents the accuracy obtainable by
predicting the more common outcome for every patient, and any useful model must exceed it by a
margin larger than the confidence interval.

In [ ]:
def classification_metrics(y_true, y_pred, y_proba):
    '''Return the full metric set for a set of predictions.'''
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
        "Sensitivity (recall)": tp / (tp + fn) if (tp + fn) else np.nan,
        "Specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "PPV (precision)": tp / (tp + fp) if (tp + fp) else np.nan,
        "NPV": tn / (tn + fn) if (tn + fn) else np.nan,
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        "Brier score": brier_score_loss(y_true, y_proba),
    }


def bootstrap_intervals(y_true, y_pred, y_proba, n_replicates=N_BOOTSTRAP, alpha=ALPHA):
    '''Percentile bootstrap confidence intervals, resampling within outcome strata.'''
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred); y_proba = np.asarray(y_proba)
    rng = np.random.default_rng(RANDOM_STATE)
    positive_idx = np.where(y_true == 1)[0]
    negative_idx = np.where(y_true == 0)[0]

    collected = {k: [] for k in classification_metrics(y_true, y_pred, y_proba)}
    for _ in range(n_replicates):
        idx = np.concatenate([
            rng.choice(positive_idx, size=len(positive_idx), replace=True),
            rng.choice(negative_idx, size=len(negative_idx), replace=True),
        ])
        for key, value in classification_metrics(y_true[idx], y_pred[idx], y_proba[idx]).items():
            collected[key].append(value)

    lower, upper = 100 * alpha / 2, 100 * (1 - alpha / 2)
    return {k: (np.nanpercentile(v, lower), np.nanpercentile(v, upper))
            for k, v in collected.items()}


DEFAULT_THRESHOLD = 0.50

# Class labels are derived by applying an explicit threshold to the predicted probability rather
# than by calling .predict(). For a support vector machine the two can disagree: .predict() uses
# the sign of the decision function, whereas predicted probabilities come from a separate Platt
# calibration fitted internally. Deriving labels from probabilities keeps every table in this
# notebook consistent with the threshold analysis in Section 23.
test_predictions, test_probabilities, test_metrics = {}, {}, {}
for name, search in fitted_searches.items():
    estimator = search.best_estimator_
    proba = estimator.predict_proba(X_test)[:, 1]
    test_probabilities[name] = proba
    test_predictions[name] = (proba >= DEFAULT_THRESHOLD).astype(int)
    test_metrics[name] = classification_metrics(y_test, test_predictions[name], proba)

baseline = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE).fit(X_train, y_train)
baseline_proba = baseline.predict_proba(X_test)[:, 1]
test_metrics["Majority-class baseline"] = classification_metrics(
    y_test, (baseline_proba >= DEFAULT_THRESHOLD).astype(int), baseline_proba
)

test_table = pd.DataFrame(test_metrics).T
order = list(nested_summary.index) + ["Majority-class baseline"]
test_table = test_table.loc[[m for m in order if m in test_table.index]]

print("Table 10. Held-out test performance, all candidates (threshold = 0.50).")
display(save_table(test_table.round(4), "table10_test_performance"))

In [ ]:
intervals = bootstrap_intervals(
    y_test, test_predictions[SELECTED_MODEL], test_probabilities[SELECTED_MODEL]
)
point = test_metrics[SELECTED_MODEL]

interval_table = pd.DataFrame({
    "Estimate": {k: point[k] for k in point},
    f"{int((1 - ALPHA) * 100)}% CI": {
        k: f"[{intervals[k][0]:.3f}, {intervals[k][1]:.3f}]" for k in point
    },
})

print(f"Table 11. Held-out test performance of the selected model ({SELECTED_MODEL}), "
      f"with {N_BOOTSTRAP:,} stratified bootstrap replicates.")
display(save_table(interval_table.round(4), "table11_selected_model_intervals"))

print(f"\nTest set: {len(y_test)} records "
      f"({int((y_test == 1).sum())} with heart disease, {int((y_test == 0).sum())} without).\n")
print(classification_report(y_test, test_predictions[SELECTED_MODEL],
                            target_names=["No heart disease", "Heart disease"],
                            zero_division=0, digits=3))

### 20.1 Corroborative comparison on the test partition

The DeLong comparison from Section 18 is repeated on the test partition. This is reported for
completeness and does not enter the selection decision, which was fixed in Section 19. Agreement
between the two comparisons strengthens the conclusion; disagreement would indicate that the
equivalence finding is sensitive to the particular partition and should be reported as such.

In [ ]:
corroboration_rows = []
for name in nested_summary.index:
    if name == SELECTED_MODEL:
        continue
    result = delong_test(y_test, test_probabilities[SELECTED_MODEL], test_probabilities[name])
    corroboration_rows.append({
        "Comparison": f"{SELECTED_MODEL} vs {name}",
        "AUC difference": result["difference"],
        f"{int((1 - ALPHA) * 100)}% CI": f"[{result['ci_low']:.4f}, {result['ci_high']:.4f}]",
        "p-value": result["p_value"],
        "Distinguishable": "Yes" if (pd.notna(result["p_value"]) and result["p_value"] < ALPHA) else "No",
    })

corroboration = pd.DataFrame(corroboration_rows).set_index("Comparison")
print("Table 12. DeLong comparisons on the held-out test partition (corroborative only).")
display(save_table(corroboration.round(4), "table12_delong_test_partition"))

agreement = all(row == "No" for row in corroboration["Distinguishable"])
print("\nThe test partition " + ("supports" if agreement else "does not fully support") +
      " the equivalence finding from the nested procedure.")

## 21. Discrimination: ROC Curves and Confusion Matrices

Confusion matrices are shown with row-normalised proportions, so that sensitivity and specificity
can be read directly from the diagonal irrespective of class sizes, with absolute counts annotated.

In [ ]:
model_names = list(nested_summary.index)
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

for ax, name in zip(axes.ravel(), model_names):
    matrix = confusion_matrix(y_test, test_predictions[name], labels=[0, 1])
    normalised = matrix / matrix.sum(axis=1, keepdims=True)
    annotations = np.array([
        [f"{normalised[i, j]:.2f}\n(n={matrix[i, j]})" for j in range(2)] for i in range(2)
    ])
    sns.heatmap(normalised, annot=annotations, fmt="", cmap="Blues", vmin=0, vmax=1,
                cbar=False, ax=ax, linewidths=0.5,
                xticklabels=["No disease", "Disease"],
                yticklabels=["No disease", "Disease"])
    marker = "  [selected]" if name == SELECTED_MODEL else ""
    ax.set_title(f"{name}{marker}", fontsize=11)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Observed")

fig.suptitle("Figure 5. Row-normalised confusion matrices on the held-out test partition "
             "(threshold = 0.50)", y=1.00)
plt.tight_layout()
save_figure("fig05_confusion_matrices")
plt.show()

In [ ]:
plt.figure(figsize=(7.5, 7))
for name in model_names:
    fpr, tpr, _ = roc_curve(y_test, test_probabilities[name])
    auc_value = roc_auc_score(y_test, test_probabilities[name])
    width = 2.4 if name == SELECTED_MODEL else 1.3
    style = "-" if name == SELECTED_MODEL else "--"
    plt.plot(fpr, tpr, style, lw=width, label=f"{name} (AUC = {auc_value:.3f})")

plt.plot([0, 1], [0, 1], ":", color="grey", lw=1, label="Chance (AUC = 0.500)")
plt.xlabel("1 - specificity (false positive rate)")
plt.ylabel("Sensitivity (true positive rate)")
plt.title("Figure 6. Receiver operating characteristic curves, held-out test partition")
plt.legend(loc="lower right", frameon=True)
plt.gca().set_aspect("equal")
plt.tight_layout()
save_figure("fig06_roc_curves")
plt.show()

## 22. Calibration

Discrimination and calibration are distinct properties. A model may rank patients correctly while
its predicted probabilities are systematically too high or too low, and the parent project intends
to display a probability to the user rather than a bare classification. Calibration is therefore
directly relevant to whether the output can responsibly be shown at all.

Two measures are reported. The calibration curve partitions predictions into bins and plots observed
outcome frequency against mean predicted probability; perfect calibration lies on the diagonal. The
Brier score is the mean squared difference between predicted probability and observed outcome, and
combines calibration and discrimination into a single value where lower is better.

Random forests trained with balanced class weighting are known to produce distorted probability
estimates, because averaging over trees pulls predictions toward the centre of the range and
reweighting shifts them further. If the selected model shows material miscalibration, isotonic or
sigmoid recalibration fitted on the training partition should be applied before deployment. That
recalibration is not performed here, as it would constitute a methodological change beyond the
approved scope.

In [ ]:
n_bins = 10
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for name in model_names:
    proba = test_probabilities[name]
    observed, predicted = calibration_curve(y_test, proba, n_bins=n_bins, strategy="quantile")
    width = 2.4 if name == SELECTED_MODEL else 1.2
    axes[0].plot(predicted, observed, "o-", lw=width,
                 label=f"{name} (Brier = {brier_score_loss(y_test, proba):.3f})")

axes[0].plot([0, 1], [0, 1], ":", color="grey", label="Perfect calibration")
axes[0].set_xlabel("Mean predicted probability")
axes[0].set_ylabel("Observed outcome frequency")
axes[0].set_title("(a) Calibration curves (quantile bins)")
axes[0].legend(fontsize=8, loc="upper left")
axes[0].set_aspect("equal")

selected_proba = test_probabilities[SELECTED_MODEL]
axes[1].hist([selected_proba[y_test == 0], selected_proba[y_test == 1]],
             bins=20, stacked=False, label=["No heart disease", "Heart disease"],
             color=["#4C72B0", "#C44E52"], alpha=0.75)
axes[1].set_xlabel("Predicted probability of heart disease")
axes[1].set_ylabel("Number of records")
axes[1].set_title(f"(b) Predicted probability distribution — {SELECTED_MODEL}")
axes[1].legend()

fig.suptitle("Figure 7. Calibration of predicted probabilities on the held-out test partition",
             y=1.02)
plt.tight_layout()
save_figure("fig07_calibration")
plt.show()

brier_table = pd.DataFrame({
    "Brier score": {name: brier_score_loss(y_test, test_probabilities[name])
                    for name in model_names}
}).sort_values("Brier score")
print("Table 13. Brier scores (lower is better).")
display(save_table(brier_table.round(4), "table13_brier_scores"))

selected_brier = brier_table.loc[SELECTED_MODEL, "Brier score"]
if selected_brier > 0.10:
    print(f"\n{SELECTED_MODEL} shows material miscalibration (Brier = {selected_brier:.4f}).")
    print("Probability recalibration is recommended before the output is displayed to a user.")
else:
    print(f"\n{SELECTED_MODEL} is reasonably calibrated (Brier = {selected_brier:.4f}).")

## 23. Decision Threshold Analysis

A probability threshold of 0.50 is the default in most software libraries but carries no clinical
authority. In a triage application the costs of the two error types are asymmetric: failing to
identify a patient with heart disease is more consequential than referring a patient without it for
further assessment.

Two alternative thresholds are derived **on the training partition's out-of-fold predictions**, so
that the operating point is not tuned on the test data:

- the threshold maximising Youden's J statistic (sensitivity + specificity − 1), which weights the
  two error types equally;
- the lowest threshold achieving at least 95% sensitivity, reflecting a screening posture in which
  false negatives are the primary concern.

Both are then applied unchanged to the test partition, and the resulting trade-offs are reported so
that the operating point can be chosen deliberately rather than by default.

In [ ]:
oof_selected = nested_oof_proba[SELECTED_MODEL]
fpr_oof, tpr_oof, thresholds_oof = roc_curve(y_train_reset, oof_selected)

youden_index = int(np.argmax(tpr_oof - fpr_oof))
threshold_youden = float(thresholds_oof[youden_index])

TARGET_SENSITIVITY = 0.95
eligible = np.where(tpr_oof >= TARGET_SENSITIVITY)[0]
threshold_sensitivity = float(thresholds_oof[eligible[0]]) if len(eligible) else 0.0

candidate_thresholds = {
    f"Default ({DEFAULT_THRESHOLD:.2f})": DEFAULT_THRESHOLD,
    f"Youden J (train-derived, {threshold_youden:.3f})": threshold_youden,
    f"Sensitivity >= {TARGET_SENSITIVITY:.0%} (train-derived, {threshold_sensitivity:.3f})":
        threshold_sensitivity,
}

threshold_rows = {}
for label, value in candidate_thresholds.items():
    predictions = (test_probabilities[SELECTED_MODEL] >= value).astype(int)
    metrics = classification_metrics(y_test, predictions, test_probabilities[SELECTED_MODEL])
    tn, fp, fn, tp = confusion_matrix(y_test, predictions, labels=[0, 1]).ravel()
    metrics.update({"False negatives": int(fn), "False positives": int(fp)})
    threshold_rows[label] = metrics

threshold_table = pd.DataFrame(threshold_rows).T.drop(columns=["ROC-AUC", "Brier score"])

print(f"Table 14. Effect of decision threshold on test performance — {SELECTED_MODEL}.")
print("Thresholds are derived from training out-of-fold predictions and applied to the test set.")
display(save_table(threshold_table.round(4), "table14_threshold_analysis"))

plt.figure(figsize=(8, 5))
grid = np.linspace(0.01, 0.99, 99)
sensitivity_curve, specificity_curve = [], []
for value in grid:
    predictions = (test_probabilities[SELECTED_MODEL] >= value).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, predictions, labels=[0, 1]).ravel()
    sensitivity_curve.append(tp / (tp + fn) if (tp + fn) else np.nan)
    specificity_curve.append(tn / (tn + fp) if (tn + fp) else np.nan)

plt.plot(grid, sensitivity_curve, label="Sensitivity", lw=2)
plt.plot(grid, specificity_curve, label="Specificity", lw=2)
for label, value in candidate_thresholds.items():
    plt.axvline(value, ls="--", lw=1, alpha=0.7)
    plt.text(value, 0.02, f" {value:.2f}", rotation=90, fontsize=8, va="bottom")
plt.xlabel("Decision threshold")
plt.ylabel("Metric value")
plt.title(f"Figure 8. Sensitivity-specificity trade-off across thresholds — {SELECTED_MODEL}")
plt.legend()
plt.tight_layout()
save_figure("fig08_threshold_tradeoff")
plt.show()

## 24. Subgroup Performance

Aggregate performance can conceal systematic differences between patient groups. A model that
performs well overall but poorly for one sex or age band would be inequitable in deployment, and
reporting subgroup performance is increasingly expected of clinical prediction models.

Subgroup sample sizes are small, so these estimates are imprecise and are reported with the number
of records in each stratum. They are intended to detect gross disparity, not to support fine
comparison.

In [ ]:
subgroup_rows = []
selected_pred = test_predictions[SELECTED_MODEL]
selected_proba = test_probabilities[SELECTED_MODEL]

for level in sorted(X_test["Sex"].dropna().unique()):
    mask = (X_test["Sex"] == level).values
    if mask.sum() >= 20 and len(np.unique(y_test[mask])) > 1:
        metrics = classification_metrics(y_test[mask], selected_pred[mask], selected_proba[mask])
        subgroup_rows.append({"Subgroup": f"Sex = {level}", "n": int(mask.sum()),
                              "Events": int(y_test[mask].sum()), **metrics})

age_bands = [(18, 45), (45, 60), (60, 200)]
for low, high in age_bands:
    mask = ((X_test["Age"] >= low) & (X_test["Age"] < high)).values
    if mask.sum() >= 20 and len(np.unique(y_test[mask])) > 1:
        label = f"Age {low}-{high - 1}" if high < 200 else f"Age >= {low}"
        metrics = classification_metrics(y_test[mask], selected_pred[mask], selected_proba[mask])
        subgroup_rows.append({"Subgroup": label, "n": int(mask.sum()),
                              "Events": int(y_test[mask].sum()), **metrics})

if subgroup_rows:
    subgroup_table = (
        pd.DataFrame(subgroup_rows).set_index("Subgroup")
        [["n", "Events", "Accuracy", "Sensitivity (recall)", "Specificity", "ROC-AUC"]]
    )
    print(f"Table 15. Subgroup performance of {SELECTED_MODEL} on the held-out test partition.")
    display(save_table(subgroup_table.round(4), "table15_subgroup_performance"))

    auc_spread = subgroup_table["ROC-AUC"].max() - subgroup_table["ROC-AUC"].min()
    print(f"\nRange of ROC-AUC across subgroups: {auc_spread:.4f}")
    if auc_spread > 0.10:
        print("Performance varies materially across subgroups. This must be reported as a")
        print("limitation and investigated before any deployment.")
    else:
        print("No gross disparity is evident, though the subgroup sample sizes are small and the")
        print("estimates correspondingly imprecise.")
else:
    print("Subgroups are too small or lack outcome variation for meaningful evaluation.")

## 25. Variable Importance

Two complementary measures are reported.

**Permutation importance** measures the reduction in ROC-AUC when a single variable's values are
randomly shuffled, breaking its association with the outcome while leaving all other variables
intact. It is computed on the held-out test partition through the complete fitted pipeline, so it
reflects the contribution of each variable as the deployed model would use it. It operates on the
original variables rather than on transformed columns, which makes it directly interpretable.

**Model-native importance** is reported alongside: absolute standardised coefficients for linear
models, or mean impurity decrease for tree-based models. These are cheaper to compute but carry
known biases — impurity-based importance systematically favours continuous and high-cardinality
variables over binary ones, which is material here because most of the strongly associated variables
are continuous laboratory measurements. Where the two measures disagree, permutation importance is
the more reliable.

Both must be read in light of the collinearity identified in Section 14. When two variables carry
overlapping information, permuting either alone understates its importance, because the model can
compensate using the other.

In [ ]:
preprocessor = FINAL_PIPELINE.named_steps["preprocess"]
classifier = FINAL_PIPELINE.named_steps["model"]
transformed_names = np.array([pretty(n) for n in preprocessor.get_feature_names_out()])

permutation_result = permutation_importance(
    FINAL_PIPELINE, X_test, y_test,
    scoring="roc_auc", n_repeats=30, random_state=RANDOM_STATE, n_jobs=-1
)

permutation_table = (
    pd.DataFrame({
        "Variable": [pretty(c) for c in X_test.columns],
        "Mean AUC decrease": permutation_result.importances_mean,
        "SD": permutation_result.importances_std,
    })
    .sort_values("Mean AUC decrease", ascending=False)
    .set_index("Variable")
)

print(f"Table 16. Permutation importance on the held-out test partition — {SELECTED_MODEL} "
      f"(30 repetitions).")
display(save_table(permutation_table.round(5), "table16_permutation_importance"))

top_permutation = permutation_table.head(15).iloc[::-1]
plt.figure(figsize=(9, 6.5))
plt.barh(top_permutation.index, top_permutation["Mean AUC decrease"],
         xerr=top_permutation["SD"], color="#4C72B0", error_kw={"lw": 1, "alpha": 0.6})
plt.xlabel("Decrease in ROC-AUC when the variable is permuted")
plt.axvline(0, color="grey", lw=1)
plt.title(f"Figure 9. Permutation importance — {SELECTED_MODEL} (test partition)")
plt.tight_layout()
save_figure("fig09_permutation_importance")
plt.show()

negligible = permutation_table[permutation_table["Mean AUC decrease"] <= 0]
print(f"\nVariables with no measurable contribution: {len(negligible)} of {len(permutation_table)}")
if len(negligible):
    print("These could be removed in a reduced model without measurable loss of discrimination.")
    print("Such a reduction is not applied here, as it would constitute a change of scope.")

In [ ]:
if hasattr(classifier, "coef_"):
    coefficients = np.asarray(classifier.coef_).ravel()
    native_table = pd.DataFrame({
        "Variable": transformed_names,
        "Coefficient": coefficients,
        "Odds ratio": np.exp(coefficients),
        "Absolute coefficient": np.abs(coefficients),
    }).sort_values("Absolute coefficient", ascending=False).set_index("Variable")

    print(f"Table 17. Standardised coefficients and odds ratios — {SELECTED_MODEL}.")
    print("Predictors are standardised, so each odds ratio expresses the multiplicative change in")
    print("the odds of heart disease per one standard deviation increase in that variable.")
    display(save_table(native_table.round(4), "table17_model_coefficients"))

    subset = native_table.head(15).iloc[::-1]
    colours = ["#C44E52" if v > 0 else "#4C72B0" for v in subset["Coefficient"]]
    plt.figure(figsize=(9, 6.5))
    plt.barh(subset.index, subset["Coefficient"], color=colours)
    plt.axvline(0, color="grey", lw=1)
    plt.xlabel("Standardised coefficient (positive increases predicted risk)")
    plt.title(f"Figure 10. Model coefficients — {SELECTED_MODEL}")
    plt.tight_layout()
    save_figure("fig10_model_coefficients")
    plt.show()

    zeroed = int((coefficients == 0).sum())
    if zeroed:
        print(f"\n{zeroed} coefficient(s) shrunk exactly to zero by L1 penalisation, indicating")
        print("that the penalty performed embedded variable selection.")

elif hasattr(classifier, "feature_importances_"):
    native_table = pd.DataFrame({
        "Variable": transformed_names,
        "Impurity importance": classifier.feature_importances_,
    }).sort_values("Impurity importance", ascending=False).set_index("Variable")

    print(f"Table 17. Mean impurity decrease — {SELECTED_MODEL}.")
    display(save_table(native_table.round(5), "table17_impurity_importance"))

    subset = native_table.head(15).iloc[::-1]
    plt.figure(figsize=(9, 6.5))
    plt.barh(subset.index, subset["Impurity importance"], color="#55A868")
    plt.xlabel("Mean decrease in impurity")
    plt.title(f"Figure 10. Impurity-based importance — {SELECTED_MODEL}")
    plt.tight_layout()
    save_figure("fig10_impurity_importance")
    plt.show()
    print("\nImpurity importance is biased toward continuous variables; compare against Table 16.")
else:
    native_table = None
    print(f"{SELECTED_MODEL} exposes no native importance measure. "
          "Permutation importance in Table 16 is the sole ranking.")

## 26. SHAP Explanation

Shapley additive explanations decompose each individual prediction into additive contributions from
each variable, derived from cooperative game theory. Unlike the global rankings in Section 25, SHAP
values are computed per observation, so they show both the magnitude and the direction of each
variable's effect, and how those effects vary across patients.

SHAP is an explanation of the fitted model, not of the underlying disease process. A variable
receiving a large SHAP value is one on which the model relies, which is not the same as a variable
that causes the outcome. Where variables are correlated, the attribution between them is to some
extent arbitrary.

In [ ]:
shap_importance = None
try:
    import shap

    X_test_transformed = preprocessor.transform(X_test)
    rng = np.random.default_rng(RANDOM_STATE)

    if isinstance(classifier, (DecisionTreeClassifier, RandomForestClassifier)):
        # TreeExplainer is exact and inexpensive; the entire test partition is explained.
        explainer = shap.TreeExplainer(classifier)
        raw = explainer.shap_values(X_test_transformed)
        explained = X_test_transformed
        if isinstance(raw, list):
            shap_values = np.asarray(raw[1])
        else:
            raw = np.asarray(raw)
            shap_values = raw[:, :, 1] if raw.ndim == 3 else raw

    elif isinstance(classifier, LogisticRegression):
        background = preprocessor.transform(
            X_train.iloc[rng.choice(len(X_train), size=min(100, len(X_train)), replace=False)]
        )
        explainer = shap.LinearExplainer(classifier, background)
        shap_values = np.asarray(explainer.shap_values(X_test_transformed))
        explained = X_test_transformed

    else:
        # Model-agnostic explanation is expensive; a stratified subsample is used.
        n_explain = min(80, len(X_test))
        background = preprocessor.transform(
            X_train.iloc[rng.choice(len(X_train), size=min(50, len(X_train)), replace=False)]
        )
        explained = X_test_transformed[:n_explain]
        explainer = shap.Explainer(
            lambda z: classifier.predict_proba(z)[:, 1], background,
            feature_names=list(transformed_names)
        )
        shap_values = np.asarray(
            explainer(explained, max_evals=max(2 * explained.shape[1] + 1, 200)).values
        )
        print(f"Model-agnostic explainer used; {n_explain} of {len(X_test)} test records explained.")

    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, -1]

    print(f"SHAP value matrix: {shap_values.shape[0]} records x {shap_values.shape[1]} features")

    plt.figure()
    shap.summary_plot(shap_values, explained, feature_names=list(transformed_names),
                      show=False, max_display=15)
    plt.title(f"Figure 11. SHAP value distribution — {SELECTED_MODEL}", fontsize=12)
    plt.tight_layout()
    save_figure("fig11_shap_summary")
    plt.show()

    shap_importance = (
        pd.DataFrame({
            "Variable": transformed_names,
            "Mean |SHAP|": np.abs(shap_values).mean(axis=0),
        })
        .sort_values("Mean |SHAP|", ascending=False)
        .set_index("Variable")
    )
    print("Table 18. Mean absolute SHAP value by variable.")
    display(save_table(shap_importance.round(5), "table18_shap_importance"))

except Exception as exc:
    print("SHAP analysis did not complete in this runtime.")
    print(f"Reason: {exc!r}")
    print("The permutation importance in Section 25 remains available as the primary explanation.")

In [ ]:
# Concordance between the explanation methods.
if shap_importance is not None:
    shap_rank = shap_importance.reset_index().assign(shap_rank=lambda t: t.index + 1)
    perm_rank = permutation_table.reset_index().assign(perm_rank=lambda t: t.index + 1)
    merged = shap_rank.merge(perm_rank[["Variable", "perm_rank"]], on="Variable", how="inner")

    if len(merged) >= 5:
        rho, p_rho = st.spearmanr(merged["shap_rank"], merged["perm_rank"])
        print("Concordance between SHAP and permutation importance rankings")
        print("-" * 66)
        print(f"Variables common to both rankings : {len(merged)}")
        print(f"Spearman rank correlation         : {rho:.4f} (p = {p_rho:.3e})")
        print("-" * 66)
        if rho > 0.7:
            print("The two methods agree closely, which strengthens confidence that the identified")
            print("variables genuinely drive the model rather than being artefacts of one method.")
        else:
            print("The two methods diverge. Divergence typically arises from correlated predictors,")
            print("among which attribution is not uniquely determined. Interpret with caution.")

        print("\nTop ten variables by each method:")
        display(pd.DataFrame({
            "By SHAP": shap_importance.index[:10].tolist(),
            "By permutation": permutation_table.index[:10].tolist(),
        }, index=range(1, 11)))
else:
    print("SHAP results unavailable; concordance analysis skipped.")

### 26.1 Interpretation

The interpretation below should be completed once the notebook has been executed, by reading it
against the tables and figures above. The following questions structure that reading and should each
be answered explicitly in the final submission:

1. **Are the leading variables clinically coherent?** Troponin-I is a direct marker of myocardial
   injury and would be expected to rank highly. Lipid fractions, glycaemic status, age and chest
   pain history are established cardiovascular risk factors. A ranking dominated by these is
   consistent with cardiovascular pathophysiology.

2. **Do any variables rank higher than their clinical role justifies?** Electrolytes and platelet
   count are not established independent predictors of coronary disease. If they rank highly, the
   most likely explanations are confounding by acuity of illness, or an artefact of the labelling
   process identified in Section 3.3.

3. **Is the direction of effect physiologically plausible?** For the coefficient-based explanation,
   higher LDL, higher triglycerides and higher troponin should increase predicted risk, and higher
   HDL should decrease it. A reversed sign on a well-established risk factor is a warning sign,
   most commonly caused by collinearity.

4. **Do SHAP and permutation importance agree?** The concordance statistic above quantifies this.

5. **Do the missingness indicators appear in the top rankings?** If they do, the informative
   missingness identified in Section 15.2 is materially influencing predictions, and the sensitivity
   analysis with `USE_MISSINGNESS_INDICATORS = False` becomes necessary rather than optional.

## 26.2 Sensitivity Analysis: Outcome-Definition Risk

Section 12.3 and Section 30.1 note that four continuous predictors — LDL cholesterol, total
cholesterol, triglycerides and haemoglobin — show univariate associations with the outcome
substantially stronger than published cardiology cohorts report for the same measurements, which is
the principal evidence that these values, rather than independent risk factors, may have contributed
to how `Heart_Disease` was originally assigned.

The data custodian who could confirm or rule out this possibility is not accessible to this project;
supervisory review here is limited to checking the work, not to providing provenance information
that is not otherwise recorded. In the absence of that confirmation, the question is addressed
empirically: the selected model architecture is refit with the four flagged variables removed, and
the resulting discrimination is compared against the full-feature model.

This does not resolve the outcome-definition question — it cannot, without the missing provenance
information — but it bounds it. If performance collapses toward chance once these variables are
removed, the full-feature result is largely an artefact of whatever produced those four
associations, circular or not. If performance remains materially above chance, the model retains
value from variables outside the flagged set, and the flagged variables' contribution, real or
circular, becomes a smaller share of the reported number.

Troponin-I is not included in the removed set here: unlike the flagged group, its univariate
association with the outcome (Section 12.3) is unremarkable (|r| = 0.07), so it is not implicated by
the evidence motivating this analysis, despite being an established clinical marker that a
circularity concern might otherwise target.

*(This addresses "Further work" item 2 in Section 31.3 for the outcome-definition question. The
analysis uses a single stratified 5-fold cross-validated grid search on the training partition, not
the full nested procedure, since it is diagnostic rather than a candidate for deployment.)*


In [ ]:
FLAGGED_VARIABLES = [
    "Total_Cholesterol(mg/dL)", "LDL(mg/dL)", "Triglycerides(mg/dL)", "Himoglobin",
]
FLAGGED_VARIABLES = [c for c in FLAGGED_VARIABLES if c in X_train.columns]

X_train_reduced = X_train.drop(columns=FLAGGED_VARIABLES)
X_test_reduced = X_test.drop(columns=FLAGGED_VARIABLES)

numeric_features_reduced = X_train_reduced.select_dtypes(include=np.number).columns.tolist()
categorical_features_reduced = X_train_reduced.select_dtypes(exclude=np.number).columns.tolist()


def build_preprocessor_reduced(scale_numeric=True):
    '''Preprocessor for the reduced feature set used in the sensitivity analysis.'''
    numeric_steps = [("impute", SimpleImputer(strategy="median",
                                              add_indicator=USE_MISSINGNESS_INDICATORS))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    categorical_pipeline = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", ONE_HOT),
    ])
    return ColumnTransformer(
        [("num", Pipeline(numeric_steps), numeric_features_reduced),
         ("cat", categorical_pipeline, categorical_features_reduced)],
        remainder="drop", verbose_feature_names_out=True,
    )


_, full_grid = model_specifications()["Random Forest"]
reduced_pipeline = Pipeline([
    ("preprocess", build_preprocessor_reduced(scale_numeric=False)),
    ("model", RandomForestClassifier(class_weight="balanced", n_jobs=-1,
                                     random_state=RANDOM_STATE)),
])

reduced_search = GridSearchCV(reduced_pipeline, full_grid, scoring="roc_auc",
                              cv=inner_cv, n_jobs=-1, refit=True)
reduced_search.fit(X_train_reduced, y_train)

reduced_cv_auc = reduced_search.best_score_
reduced_test_proba = reduced_search.best_estimator_.predict_proba(X_test_reduced)[:, 1]
reduced_test_auc = roc_auc_score(y_test, reduced_test_proba)

sensitivity_table = pd.DataFrame({
    "Full feature set": [nested_summary.loc[SELECTED_MODEL, "Nested ROC-AUC (pooled)"],
                         test_metrics[SELECTED_MODEL]["ROC-AUC"]],
    "Flagged variables removed": [reduced_cv_auc, reduced_test_auc],
}, index=["Cross-validated ROC-AUC (training)", "Held-out test ROC-AUC"])

print(f"Variables removed: {FLAGGED_VARIABLES}")
print(f"Remaining predictors: {X_train_reduced.shape[1]}")
print()
print("Table 16b. Sensitivity analysis: Random Forest with and without the flagged")
print("lipid/haemoglobin variables.")
display(save_table(sensitivity_table.round(4), "table16b_sensitivity_outcome_definition"))

auc_change = (sensitivity_table.loc["Held-out test ROC-AUC", "Flagged variables removed"]
             - sensitivity_table.loc["Held-out test ROC-AUC", "Full feature set"])
print(f"\nTest ROC-AUC change when flagged variables are removed: {auc_change:+.4f}")

if reduced_test_auc > 0.75:
    print(
        "The model retains materially above-chance discrimination without the flagged variables.\n"
        "This indicates the flagged variables are not solely responsible for the reported\n"
        "performance, though their individual contribution cannot be characterised as circular or\n"
        "genuine without the outcome-definition information that could not be obtained for this\n"
        "project. Report the full-feature result as bounded by both figures in Table 16b, not as\n"
        "an unqualified single AUC value."
    )
else:
    print(
        "Discrimination falls sharply without the flagged variables, consistent with the\n"
        "full-feature performance being substantially dependent on them. Given the unresolved\n"
        "outcome-definition question, the full-feature result should not be reported as evidence of\n"
        "diagnostic accuracy."
    )


## 27. Worked Example

A single test record is passed through the complete fitted pipeline to demonstrate the inference
path from raw input to predicted probability. This illustrates the interface the parent project's
web application would use: unprocessed variables in, calibrated probability out, with all
imputation, encoding and scaling handled internally by the saved pipeline.

In [ ]:
example_index = 0
example = X_test.iloc[[example_index]]
example_probability = float(FINAL_PIPELINE.predict_proba(example)[0, 1])
example_class = int(example_probability >= DEFAULT_THRESHOLD)
observed_class = int(y_test.iloc[example_index])

presented = example.T.rename(columns={example.index[0]: "Value"})
presented.index = [pretty(i) for i in presented.index]

print("Input record (as supplied, before any preprocessing):")
display(presented)

print("\nModel output")
print("-" * 58)
print(f"Predicted probability of heart disease : {example_probability:.4f}")
print(f"Predicted class at threshold 0.50      : {example_class}")
print(f"Observed outcome                       : {observed_class}")
print(f"Missing values handled internally      : {int(example.isna().sum(axis=1).iloc[0])}")
print("-" * 58)
print("This is the output of a research prototype. It is not a diagnosis and must not be used")
print("to inform the care of any individual patient.")

## 28. Saved Artifacts and Reproducibility Record

The fitted pipeline is saved in a form that accepts raw, unprocessed input, so that a downstream
application need not reimplement any preprocessing step. A metadata record captures the resolved
library versions, the source file digest, the partition and cross-validation configuration, the
selection outcome and the derived thresholds — everything required to determine, at a later date,
exactly what produced a given set of results.

> **Data governance.** The cleaned modelling dataset written below contains patient-level clinical
> records. It must not be published, shared outside the approved research team, or uploaded to a
> public repository without written authorisation from the data custodian and the approving ethics
> committee. Only the model artifact, the aggregate result tables and the figures are suitable for
> distribution.

In [ ]:
MODEL_PATH = os.path.join(OUTPUT_DIR, "heart_disease_inference_pipeline.joblib")
METADATA_PATH = os.path.join(OUTPUT_DIR, "model_metadata.json")
DATASET_PATH = os.path.join(OUTPUT_DIR, "RESTRICTED_modelling_dataset.csv")

joblib.dump(FINAL_PIPELINE, MODEL_PATH)
df_model.to_csv(DATASET_PATH, index=False)

metadata = {
    "project": "AI-Driven Web-Based Heart Disease Prediction System Using Machine Learning",
    "notebook": "Northern Bangladesh heart disease classification",
    "environment": ENVIRONMENT,
    "data": {
        "source_file": os.path.basename(DATA_PATH),
        "sha256": DATA_SHA256,
        "sheet": SHEET,
        "records_supplied": int(len(df_raw)),
        "records_modelled": int(len(df_model)),
        "paediatric_excluded": int(paediatric.sum()),
        "duplicate_clinical_records": dup_clinical,
        "excluded_variables": EXCLUDED_VARIABLES,
        "predictors": X.columns.tolist(),
        "outcome": "Heart_Disease",
    },
    "configuration": {
        "test_size": TEST_SIZE,
        "outer_folds": N_OUTER_FOLDS,
        "inner_folds": N_INNER_FOLDS,
        "bootstrap_replicates": N_BOOTSTRAP,
        "alpha": ALPHA,
        "drop_maxhr": DROP_MAXHR,
        "use_missingness_indicators": USE_MISSINGNESS_INDICATORS,
        "missingness_only_auc": None if np.isnan(indicator_auc) else round(indicator_auc, 4),
    },
    "selection": {
        "candidates": list(nested_summary.index),
        "reference_model": reference_model,
        "equivalence_set": equivalent_models,
        "selected_model": SELECTED_MODEL,
        "selection_rule": "Highest nested CV AUC; most parsimonious model not distinguishable "
                          "from it by DeLong test",
        "hyperparameters": {k: str(v) for k, v in FINAL_PARAMS.items()},
    },
    "performance": {
        "nested_cv": nested_summary.round(4).to_dict(orient="index"),
        "held_out_test": {k: round(float(v), 4) for k, v in test_metrics[SELECTED_MODEL].items()},
        "test_confidence_intervals": {
            k: [round(float(lo), 4), round(float(hi), 4)] for k, (lo, hi) in intervals.items()
        },
    },
    "thresholds": {
        "default": 0.50,
        "youden_train_derived": round(threshold_youden, 4),
        "sensitivity_target_train_derived": round(threshold_sensitivity, 4),
        "target_sensitivity": TARGET_SENSITIVITY,
    },
    "disclaimer": "Research prototype. Not externally validated, not prospectively evaluated, "
                  "and not approved for clinical use.",
}

with open(METADATA_PATH, "w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2, default=str)

print("Saved artifacts")
print("-" * 70)
for path in [MODEL_PATH, METADATA_PATH, DATASET_PATH]:
    print(f"  {path}  ({os.path.getsize(path) / 1024:.1f} KB)")
print(f"  {FIG_DIR}/  ({len(os.listdir(FIG_DIR))} files)")
print(f"  {TAB_DIR}/  ({len(os.listdir(TAB_DIR))} files)")
print("-" * 70)

# Verify the saved pipeline reproduces the in-memory model exactly.
reloaded = joblib.load(MODEL_PATH)
assert np.allclose(reloaded.predict_proba(X_test)[:, 1], test_probabilities[SELECTED_MODEL]), \
    "Saved pipeline does not reproduce in-memory predictions."
print("Saved pipeline verified: predictions reproduce exactly on reload.")

## 29. Summary of Results

In [ ]:
selected_metrics = test_metrics[SELECTED_MODEL]
nested_row = nested_summary.loc[SELECTED_MODEL]

print("=" * 74)
print("HEART DISEASE CLASSIFICATION - NORTHERN BANGLADESH COHORT")
print("=" * 74)
print(f"{'Records supplied':<38} {len(df_raw)}")
print(f"{'Records modelled (adults)':<38} {len(df_model)}")
print(f"{'Candidate predictors':<38} {X.shape[1]}")
print(f"{'Training / held-out test':<38} {len(X_train)} / {len(X_test)}")
print(f"{'Outcome prevalence (full cohort)':<38} {y.mean():.3f}")
print("-" * 74)
print(f"{'Selected model':<38} {SELECTED_MODEL}")
print(f"{'Selection rule':<38} nested CV AUC, then parsimony under equivalence")
print(f"{'Equivalence set':<38} {', '.join(equivalent_models)}")
print("-" * 74)
print("NESTED CROSS-VALIDATION (training partition, unbiased)")
print(f"{'  ROC-AUC (mean of outer folds)':<38} "
      f"{nested_row['Nested ROC-AUC (mean)']:.4f} (SD {nested_row['Nested ROC-AUC (SD)']:.4f})")
print(f"{'  Sensitivity':<38} {nested_row['Nested sensitivity']:.4f}")
print(f"{'  Specificity':<38} {nested_row['Nested specificity']:.4f}")
print("-" * 74)
print(f"HELD-OUT TEST PARTITION (n = {len(y_test)}, threshold 0.50, 95% CI)")
for metric in ["ROC-AUC", "Accuracy", "Balanced accuracy", "Sensitivity (recall)",
               "Specificity", "PPV (precision)", "NPV", "F1", "Brier score"]:
    low, high = intervals[metric]
    print(f"{'  ' + metric:<38} {selected_metrics[metric]:.4f}  [{low:.3f}, {high:.3f}]")
print("-" * 74)
print(f"{'Majority-class baseline accuracy':<38} "
      f"{test_metrics['Majority-class baseline']['Accuracy']:.4f}")
print("=" * 74)
print("Research prototype. Not validated for clinical use. See Section 30 for limitations.")
print("=" * 74)

## 30. Limitations

### 30.1 Outcome definition and the plausibility of the reported performance

The most significant limitation concerns the interpretation of the outcome variable. The
associations between individual routine laboratory measurements and the outcome, reported in Section
12.3, are substantially stronger than published cohort studies report for the same measurements. If
lipid, glycaemic or troponin values contributed to the clinical determination of heart disease
status, then the model is in part recovering the diagnostic rule that generated the labels rather
than predicting disease from independent evidence.

This possibility does not invalidate the methodology, which would be applied identically to a
cleanly labelled dataset. It does bear directly on what the reported performance means. The outcome
definition requested in Section 3.3 could not be obtained for this project — the data custodian is
not accessible, and supervisory review does not extend to supplying that information. This is
reported here as an unresolved limitation rather than assumed away in either direction.

In its place, Section 26.2 reports an empirical sensitivity analysis: the selected model refit
without the four variables implicated by Section 12.3 (LDL cholesterol, total cholesterol,
triglycerides, haemoglobin). The result (Table 16b) bounds, but does not eliminate, the risk —
materially above-chance discrimination without the flagged variables would indicate the full-feature
result is not solely an artefact of the flagged associations, while a collapse toward chance would
indicate the opposite. Whichever outcome obtained, the headline performance figures in Section 29
must be read alongside Table 16b and this caveat, and should not be reported as evidence of
diagnostic accuracy on their own.

### 30.2 Single-centre data without external validation

All records originate from a single geographic region and, so far as is documented, a single
institution. The held-out partition provides an estimate of performance on unseen patients from the
same source, but not of transportability to another hospital with different equipment, referral
patterns, case mix or laboratory calibration. Performance on external data is typically lower, often
substantially so. External validation on an independent cohort is the necessary next step before any
claim of generalisability.

### 30.3 Sample size

With 1,035 modelled records and 207 in the held-out partition, confidence intervals on the test
metrics are wide, and the subgroup analyses in Section 24 are indicative rather than conclusive. The
events-per-variable ratio is adequate for the modelling undertaken, but the sample does not support
fine-grained comparison between models or reliable estimation of performance within small strata.

### 30.4 Cross-sectional design

The data represent a single time point per record. The model classifies prevalent disease status
rather than predicting future events, and nothing here supports inference about incidence, prognosis
or the effect of intervention. Temporal ordering between the laboratory measurements and the
diagnosis is not documented.

### 30.5 Missing data

Missingness ranges up to approximately 9% across variables and is handled by median imputation
within the pipeline. Median imputation assumes values are missing at random conditional on the
observed data — an assumption that is unlikely to hold exactly in clinical records, where the
decision to order a test is itself clinically driven. The diagnostic in Section 15.2 quantifies the
extent to which missingness carries outcome information. Multiple imputation would propagate
imputation uncertainty into the reported intervals and would be a methodological improvement.

### 30.6 Measurement heterogeneity

Troponin-I is measured on two assay platforms and harmonised by scale conversion. This assumes the
platforms are directly comparable after unit conversion, which is an approximation: assays differ in
antibody specificity and in their reported reference limits. Censored values are substituted rather
than modelled, and the substitution convention, though standard, introduces a known bias in the
tails. Units for haemoglobin, potassium, chloride, platelet count and maximum heart rate are inferred
from value ranges rather than documented, and the blood pressure variable's definition is
unspecified.

### 30.7 Derived and collinear variables

The diagnostic in Section 10 examines whether maximum heart rate is measured or derived from age. If
derived, its apparent contribution is an artefact. Body mass index is a deterministic function of
height and weight, all three of which are retained. Correlated variables complicate the attribution
of importance in Sections 25 and 26, as the contribution of a variable can be absorbed by its
correlates.

### 30.8 Absence of clinical variables of established value

The dataset contains no electrocardiographic findings, no echocardiographic measurements, no
angiographic data, no smoking status, no medication history and no symptom characterisation beyond a
binary chest pain indicator. A model without these variables cannot be compared directly with
established clinical risk scores, and its performance should not be interpreted as an assessment of
what a complete clinical workup would achieve.

### 30.9 Fairness and equity

Subgroup performance is reported for sex and age band only. Socioeconomic status, rural or urban
residence and ethnicity are not recorded, so disparities along those dimensions cannot be assessed.
The cohort is drawn from patients who reached a hospital, which excludes those who did not — a
selection mechanism likely to correlate with the same factors.

### 30.10 Deployment considerations not addressed

Calibration is assessed but recalibration is not applied. No prospective evaluation, usability
assessment, or study of how a clinician or patient would act on the model's output has been
conducted. Model drift monitoring, an audit trail and a defined mechanism for clinical override
would all be prerequisites for any deployment.

## 31. Conclusion and Clinical Disclaimer

### 31.1 Summary

This notebook develops and internally validates a supervised classifier for heart disease status
using routinely collected admission data from a northern Bangladesh hospital cohort. Four
classifiers spanning distinct inductive biases were evaluated under an identical leakage-controlled
preprocessing pipeline. Performance was estimated by nested cross-validation, which separates
hyperparameter selection from performance estimation and avoids the optimistic bias that arises when
a tuned configuration is scored on the folds used to select it. Candidate models were compared
statistically by the DeLong test on out-of-fold predictions, and the final model was chosen by a
rule fixed in advance that prefers the more parsimonious model where discriminative performance
cannot be distinguished. The held-out partition was reserved for final reporting and played no part
in any selection decision.

Performance is reported with bootstrap confidence intervals, at multiple decision thresholds, with
calibration assessed alongside discrimination, and with subgroup results reported for sex and age.
The model is explained by two independent methods whose agreement is quantified.

### 31.2 Principal caveat

The strength of the association between individual laboratory measurements and the outcome exceeds
what the clinical literature reports, and the outcome definition could not be documented for this
project — the data custodian is not reachable, and supervisory review here is limited to checking
the completed work rather than supplying provenance. Section 26.2's sensitivity analysis is reported
in place of that confirmation: it shows how much discrimination survives removal of the four
implicated variables (Table 16b), which bounds but does not close the question. The reported
performance should be characterised as the model's ability to reproduce recorded heart disease
status in this dataset, bounded by the sensitivity result, rather than as an unqualified measure of
diagnostic accuracy. This caveat should accompany every statement of the results.

### 31.3 Further work

1. Obtain the outcome definition and adjudication procedure from the data custodian, and confirm
   that records represent unique patients. This could not be done within this project; Section 26.2
   reports the empirical substitute adopted instead, and the gap remains open for any future work
   that can reach the custodian.
2. The derived-variable (Section 10) and informative-missingness (Section 15.2) questions have been
   evaluated: `MaxHR` shows substantial but not conclusive age-dependence and is retained pending
   confirmation, and missingness carries borderline outcome information (CV-AUC 0.636). The
   outcome-definition sensitivity analysis (Section 26.2) has likewise been run; its result should be
   re-examined if the flagged variables' provenance is ever clarified.
3. Conduct external validation on an independent cohort from a different institution.
4. Apply probability recalibration and evaluate its effect on the Brier score, if the calibration
   assessment indicates it is required.
5. Replace single imputation with multiple imputation so that imputation uncertainty is reflected in
   the reported intervals.
6. Complete a TRIPOD+AI reporting checklist for the manuscript.

---

### Clinical disclaimer

> The model developed in this notebook is a **research prototype**. It has not been externally
> validated, prospectively evaluated, clinically trialled, or assessed by any regulatory authority.
> It is not a medical device and is not a diagnostic tool.
>
> No output of this model should be used to inform the diagnosis, treatment or management of any
> individual patient. The analysis characterises statistical association within a single
> retrospective dataset; association is not diagnosis, and the model has no capacity to assess a
> patient it has not been given data about, to recognise a presentation unlike those in its training
> data, or to know when it is wrong.
>
> Any future clinical application would require external validation, prospective evaluation,
> regulatory assessment and clinical governance appropriate to the jurisdiction of use.